## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [12]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [13]:
# Always remember to do this!
load_dotenv(override=True)

True

In [14]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AI
DeepSeek API Key not set (and this is optional)
Groq API Key not set (and this is optional)


In [15]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [16]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [17]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


You're the autonomous coordinator of 3 ambulances after a major earthquake; you must allocate and schedule them for the next 90 minutes to maximize expected lives saved subject to legal/ethical constraints (no prioritization by protected attributes) and robustness to uncertain travel delays and possible misinformation in reports. Current situation: Site A reports 12 injured (3 critical, 9 non-critical); Site B reports 20 injured (6 critical, 14 non-critical); Site C reports 40 injured (10 critical, 30 non-critical). Survival probabilities depend on ambulance arrival time: if an ambulance arrives within 10 minutes, critical/non-critical survival probabilities are A: (0.8/0.95), B: (0.7/0.90), C: (0.6/0.85); if arrival is between 10–30 minutes they become A: (0.4/0.80), B: (0.3/0.70), C: (0.2/0.60); if arrival is after 30 minutes assume survival = 0.05 for critical and 0.20 for non‑critical. Travel times from the depot have baselines t_A=8 min, t_B=12 min, t_C=20 min, but each site’s tra

In [18]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [19]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Below is a concrete plan and a robust accompanying analysis. I’ll keep the presentation concrete (numbers, timings, decisions) while also noting assumptions, sensitivities, and how you’d run this in real time.

1) Concrete allocation and 90‑minute trip schedule (who goes where and when)

Assumptions used for scheduling
- All ambulances start at the depot at time 0.
- Each trip is to a single site, with a return to the depot before starting another trip.
- Each trip can treat up to 3 patients (or as many as are available at the site).
- Travel time baseline t at site: A=8 min, B=12 min, C=20 min.
- Independent delay D with P(D>5)=0.3 and P(D>15)=0.05 per trip; we use the expected delay E[D] ≈ 5.25 min (baseline calculation: mix of intervals 0–5, 5–15, >15; E[D] ≈ 2.5*0.7 + 10*0.25 + 20*0.05 ≈ 5.25). Each trip’s one‑way time becomes baseline + E[D].
- Per‑trip round‑trip travel time ≈ 2 × (baseline + E[D]).
- Reports are uncertain: with probability 0.20 a site’s counts are overstated by 50%. We use the follow‑through assumption that overstated means actual counts are reported/1.5; otherwise actual = reported. Across sites, overstated events are independent.
- Survey/triage logic: we allocate trips to locations with remaining patients, prioritizing earlier arrivals for sites where earlier arrival affects survival prospects (i.e., earlier windows still matter even if we know arrival time won’t be in the 0–10 min bracket for some sites).
- Survival probabilities by arrival window and patient type (critical vs non‑critical) are as given in the problem.

Concrete allocation (best straightforward use of 3 ambulances over next 90 minutes)
- Trip plan (three ambulances: A1, B1, C1; then A2, B2; then A3)
- Site counts used for planning (expected counts, accounting for 0.20 misreport probability):
  - Site A: reported 12 with 3 critical / 9 non‑critical. If overstated (0.20), actuals scale to 12/1.5 ≈ 8 with 2 critical and 6 non‑critical (proportions preserved). If not overstated, 12 with 3 critical and 9 non‑critical. We’ll operate with the expected mix around 10–12 total for planning but treat 3 per trip so as not to run out prematurely.
  - Site B: reported 20 (6 critical / 14 non‑critical). If overstated, ≈ 13.3 total (≈4/9 critical/non‑critical mix of the same proportion). Planning uses 6 per three‑trip window.
  - Site C: reported 40 (10 critical / 30 non‑critical). If overstated, ≈ 26.7 total (≈ 7/19 critical/non). Planning uses 3 per trip.

- Schedule (times use one‑way travel times plus delays; round‑trip times approximate; all times are wall clock minutes from 0):
  - Ambulance A1 → Site A (0 to ~26.5 minutes total): depart 0; one‑way time ≈ 8 + 5.25 ≈ 13.25; round trip ≈ 26.5; arrive at Site A around t ≈ 13–14; return around t ≈ 26–27. Treat up to 3 patients here.
  - Ambulance B1 → Site B (0 to ~34.5 minutes total): depart 0; one‑way ≈ 12 + 5.25 ≈ 17.25; round trip ≈ 34.5; arrive around t ≈ 17–18; return around t ≈ 34–35. Treat up to 3 patients here.
  - Ambulance C1 → Site C (0 to ~50.5 minutes total): depart 0; one‑way ≈ 20 + 5.25 ≈ 25.25; round trip ≈ 50.5; arrive around t ≈ 25–26; return around t ≈ 50–51. Treat up to 3 patients here.
  - Ambulance A2 → Site A (after A1 finishes around t ≈ 26–27): depart ~t=26.5; one‑way ≈ 13.25; arrive ≈ t ≈ 39–40; return ≈ t ≈ 52–53. Treat up to 3 patients here (second trip to A).
  - Ambulance B2 → Site B (after B1 finishes around t ≈ 34–35): depart ~t=34.5; one‑way ≈ 17.25; arrive ≈ t ≈ 51–52; return ≈ t ≈ 68–69. Treat up to 3 patients here (second trip to B).
  - Ambulance A3 → Site A (after A2 finishes around t ≈ 52–53): depart ~t=52.5; one‑way ≈ 13.25; arrive ≈ t ≈ 65–66; return ≈ t ≈ 79–80. Treat up to 3 patients here (third trip to A).

- Rationale for allocations
  - It maximizes number of trips within 90 minutes, given round‑trip times. A can do up to three trips in this window, B up to two, C only one (under these travel‑delay assumptions). This allocation concentrates early availability at Site A (fastest baseline travel time) and still provides one early pass to each of B and C to capture higher survival probabilities in their 10–30 minute window, then uses the remaining time to complete a third A trip where most remaining patients are still at risk but arrival windows are less optimal.

- Summary of trips and who goes where
  - A1: 0–~26.5, Site A, 3 patients
  - B1: 0–~34.5, Site B, 3 patients
  - C1: 0–~50.5, Site C, 3 patients
  - A2: ~26.5–~52.5, Site A, 3 patients
  - B2: ~34.5–~69, Site B, 3 patients
  - A3: ~52.5–~79.5, Site A, 3 patients

Notes
- If Site A or B runs out of patients earlier than assumed, adjust by reassigning the spare capacity to other sites (most likely to C) to avoid wasted trip time.
- We are ignoring on‑site treatment time beyond “brief” intake; if treatment time is longer, reduce the number of trips accordingly.

2) Computed expected lives saved under the stated uncertainties and assumptions

Key inputs used (rounded)
- Site counts (expected, given misreports):
  - A: about 11.2 total injured (2–3 critical, 6–9 non‑critical per overstated/not overstated mix)
  - B: about 18.7 total (roughly 30% critical)
  - C: about 37.3 total (roughly 25% critical)
- Per‑trip patient distribution (assuming each site contributes roughly 3 patients per trip, and 3 trips to A, 2 to B, 1 to C):
  - Site A per trip (3 patients): expected critical ≈ 0.75, non‑critical ≈ 2.25
  - Site B per trip (3 patients): expected critical ≈ 0.9, non‑critical ≈ 2.1
  - Site C per trip (3 patients): expected critical ≈ 0.75, non‑critical ≈ 2.25
- Survival by arrival window and patient type (as given):
  - Arrival 0–10 min: A: 0.8/0.95 (critical/non), B: 0.7/0.9, C: 0.6/0.85
  - Arrival 10–30 min: A: 0.4/0.8, B: 0.3/0.7, C: 0.2/0.6
  - Arrival >30 min: all: critical 0.05, non-critical 0.20

Arrival windows for the six trips (using the base delay distribution E[D] ≈ 5.25)
- A1 arrival ≈ 13.25 min → 10–30 window
- B1 arrival ≈ 17.25 min → 10–30 window
- C1 arrival ≈ 25.25 min → 10–30 window
- A2 arrival ≈ 39–40 min → >30 window
- B2 arrival ≈ 51–52 min → >30 window
- A3 arrival ≈ 65–66 min → >30 window

Per‑trip expected lives saved (3 patients per trip, using the above per‑trip expectations)
- A1 (10–30 window): 3 patients with 0.75 critical and 2.25 non‑critical
  - EV_A1 = 0.75 × 0.4 + 2.25 × 0.8 = 0.30 + 1.80 = 2.10
- B1 (10–30 window): 3 patients with 0.9 critical and 2.1 non‑critical
  - EV_B1 = 0.9 × 0.3 + 2.1 × 0.7 = 0.27 + 1.47 = 1.74
- C1 (10–30 window): 3 patients with 0.75 critical and 2.25 non‑critical
  - EV_C1 = 0.75 × 0.2 + 2.25 × 0.6 = 0.15 + 1.35 = 1.50
- A2 (>30 window): 3 patients with 0.75 critical and 2.25 non‑critical
  - EV_A2 = 0.75 × 0.05 + 2.25 × 0.20 = 0.0375 + 0.45 = 0.4875
- B2 (>30 window): EV_B2 = 0.75 × 0.05 + 2.25 × 0.20 = 0.4875
- A3 (>30 window): EV_A3 = 0.4875

Sum of the six trips’ expected lives saved
- Total EV ≈ 2.10 + 1.74 + 1.50 + 0.4875 + 0.4875 + 0.4875 ≈ 6.80 lives

Thus, under the baseline uncertainty model and the proposed six-trip plan, the expected lives saved over the next 90 minutes is about 6.8 lives (rounded; note this is a probabilistic expectation because of uncertainty in site counts and travel delays).

3) Clear list of modeling assumptions and sensitivity

Assumptions (and likely sensitivities)
- Travel delays model:
  - D is independent across trips with P(D>5)=0.3 and P(D>15)=0.05; we use E[D] ≈ 5.25.
  - Sensitivity: If delays are consistently shorter or longer (e.g., P(D>5)=0.1 or 0.5), arrival windows change. If A or B could reach within 0–10 min by some improbable D realization, some survival probabilities would improve (e.g., critical survival from 0.8→0.4–0.8 range). In the current plan, none of the six trips reliably land in 0–10 min; improvements here have limited effect unless D reductions are large.
- Counts and misreporting:
  - Each site overstated with probability 0.20; overstated means actual counts = reported/1.5 with the same ratio of critical to non‑critical (approximately). We used expected totals across sites to guide staffing; the actual counts could be higher or lower within a modest range.
  - Sensitivity: If misreport probability p is 0.40, the expected counts drop further (A ≈ 10.4, B ≈ 17.3, C ≈ 34.7). With the six‑trip plan (needing 9 A patients, 6 B, 3 C across the six trips), you’re still typically not capacity‑limited by C or B, but A could be closer to shortage if counts fall much more.
- Trip capacity and time:
  - Each trip handles up to 3 patients; you can carry fewer if the site has fewer remaining patients. If site counts drop below the planned 3 per trip, the per‑trip EV scales down accordingly.
  - No on‑scene treatment time is explicitly modeled (assume minimal).
- Survival probabilities:
  - The three labeled windows (0–10, 10–30, >30) and the associated survival for critical and non‑critical are fixed. If clinical reality shows different survival curves (e.g., faster deteriorations), the EV per trip will change.
- Independence:
  - Events are treated as independent across sites and trips. In the real world, congestion, road closures, and resource saturation can correlate travel times and survival outcomes.
- Ethics and legal:
  - We are not prioritizing by protected attributes; all prioritization is based on need (counts and survivability). Triage decisions are subject to medical ethics guidelines and local laws.

How sensitive is the plan?
- High sensitivity: If total available patients across all sites drop below the cumulative demand of the six trips, you’ll waste trips or leave some patients untreated; reallocation would then be required.
- Medium sensitivity: Shifts in travel delays and arrival windows move some trips across windows (e.g., a trip moving from 10–30 to >30) which reduces the expected EV per trip.
- Low sensitivity: If misreport probability changes modestly from 0.20 to 0.15 or 0.40 but still leaves enough patients at each site to fill all six trips, overall EV remains roughly in the same ballpark; the main effect is a slight scaling of EV with available patients.

4) Two alternate priors and their impact on expected lives saved

(4a) Travel delays are more optimistic (P(D>5)=0.1, P(D>15)=0.01)
- Recompute E[D]:
  - P(D≤5) ≈ 0.9
  - P(5<D≤15) ≈ 0.08
  - P(D>15) ≈ 0.01
  - E[D] ≈ 0.9×2.5 + 0.08×10 + 0.01×20 ≈ 2.25 + 0.80 + 0.20 ≈ 3.25 minutes
- Therefore one‑way travel times shrink modestly:
  - A1 one‑way ≈ 11.25 min; round trip ≈ 22.5
  - B1 one‑way ≈ 15.25 min; round trip ≈ 30.5
  - C1 one‑way ≈ 23.25 min; round trip ≈ 46.5
- Arrival windows for the six trips (A1, B1, C1, A2, B2, A3) remain in 10–30 for A1, B1, C1; A2/A3 and B2 move further toward >30 but still within time windows enabling 3‑pax treatment per trip.
- Expected lives saved with the same trip plan stay about the same (the per‑trip EVs for first three trips remain 2.10, 1.74, 1.50, and subsequent trips remain 0.4875 or 0.465 depending on bracket). Resulting total EV ~ 6.8 lives (similar to baseline). So, more optimistic delays have little qualitative effect given the arrival windows observed here.

(4b) Misinformation probability is 0.40 (higher chance of overstated reports)
- Recompute expected counts with p_overstate = 0.40. Overstated probability means actual = reported/1.5 with the same critical/non‑critical proportions.
  - Site A: reported 12 → if overstated: 8 total (2 critical, 6 non‑critical); otherwise 12 (3 critical, 9 non‑critical). Expectation skews downward: E[total A] ≈ 0.40×8 + 0.60×12 = 3.2 + 7.2 = 10.4
  - Site B: reported 20 → overstated: ≈ 13.3 total; Expectation E[total B] ≈ 0.40×13.3 + 0.60×20 ≈ 5.3 + 12 = 17.3
  - Site C: reported 40 → overstated: ≈ 26.7 total; E[total C] ≈ 0.40×26.7 + 0.60×40 ≈ 10.7 + 24 = 34.7
- With six trips planned (nine to A, six to B, three to C capacity across three ambulances), these counts still exceed the need for six trips, so you won’t run out of patients in this six‑trip plan. The expected lives saved remains around 6.7–6.8 (slightly lower than baseline if you were at risk of hitting a site with very low remaining counts, but in this schedule you’re usually not capacity constrained).
- If, in a different layout, one site were to be exhausted before completing its planned three trips, the higher misreport rate would further reduce EV because trips would be wasted on empty capacity. In the current plan, the effect is small.

Bottom line for (4): 
- (a) More optimistic delays: essentially no meaningful change in total EV for this plan (≈6.8 lives).
- (b) Misreport probability 0.40: modest potential reduction in EV if counts become binding; with the six‑trip plan shown, EV remains near ≈6.7–6.8 lives unless counts become binding at one site.

5) A robust real‑time decision algorithm or pseudocode (including belief updates and reallocation on new drone data)

Idea: A belief‑driven, event‑driven, reallocation algorithm with a rolling horizon (60–90 minutes). Use Monte Carlo samples to account for travel delays and counts, and reallocate when new information arrives.

Pseudocode (high level)

- State:
  - Ambulances: A1, A2, A3 with their next available time and current target site (or idle at depot)
  - Site beliefs: for each site s ∈ {A, B, C}, a distribution over remaining critical and non‑critical counts. Initialize from reported numbers and the overstated probability p_overstate = 0.2 (or 0.4 if you’re testing alt priors).
  - Time t = 0
  - Observations queue: drone surveys, updates to counts, weather/road information, etc.

- Initialization:
  - Set initial counts per site (expected counts) using p_overstate (baseline 0.2) or your chosen prior.
  - For each trip to be considered, precompute per‑site per‑trip value:
    - Given arrival window (depends on t and random D), compute expected lives for a 3‑patient trip using the current expected mix at that site and the survival probabilities by arrival window.
  - Create a simple priority metric for each potential trip t_trip(s) = EV_trip(s, arrival_time) / travel_time_effort. Here EV_trip is the expected lives from treating up to 3 patients at site s, given their mix and the arrival window.

- Decision loop (event‑driven):
  - Whenever an ambulance becomes available at time t_avail (initially t_avail = 0 for all 3):
    - For each site s with remaining patients > 0, sample D for a trip (or compute expectation using the D distribution) to estimate arrival_time ≈ t_avail + 2 × (baseline_s + E[D]).
    - Determine arrival window for that trip (0–10, 10–30, >30) from arrival_time.
    - Compute EV_trip(s) given the arrival window and current site composition (critical vs non‑critical mix). Cap to remaining patients (min(3, remaining at s)).
    - Compute the best s with maximum EV_trip(s) per trip and assign this ambulance to that site.
    - Update states: mark ambulance as busy until t_avail + 2 × (baseline_s + D_sample); deduct 3 patients from site (or fewer if fewer remain) once trip completes (or apply expectation if you want probabilistic accounting).
  - When a drone or a new data feed arrives:
    - Update beliefs for each site, recalculate EV_trip for upcoming opportunities.
    - If the optimal next target for one or more ambulances changes, reallocate immediately (preemptively only if you believe it’s safe to reassign during a trip – otherwise, wait until next available time to avoid dangerous mid‑flight changes). In practice, use a policy: only reallocate at trip boundaries to avoid mid‑mission disruptions.
  - End condition: t reaches 90 minutes.

- Belief update on drone data:
  - If drone shows site s has fewer patients than believed, reduce remaining count for s accordingly. If more, increase remaining count accordingly (subject to physical plausibility and maximum counts).
  - If counts change, run the EV_trip recalculation for all three ambulances and reallocate as above.

- Practical notes for implementation:
  - Use a small number of Monte Carlo samples (e.g., 200–1000) to estimate EV_trip under D randomness for each potential trip, per site, per arrival time.
  - Maintain a rolling window of the next 3–6 trips to keep planning tractable.
  - Implement guardrails to ensure no protected attributes are used in triage; decisions are based on need and survival probability windows only.
  - Always allow manual override by a human clinician/surgeon in the loop.

6) Simulation/stress tests and metrics to evaluate robustness and fairness

Three+ tests you would run
- Test 1: Travel delay stress test
  - Vary D distribution across three regimes: pessimistic (P(D>5)=0.5, P(D>15)=0.5), baseline, optimistic (P(D>5)=0.1, P(D>15)=0.01).
  - Metrics: expected lives saved, distribution of arrival windows by site, number of patients treated per site, total trips completed.
  - Goal: confirm plan remains near optimal and robust to travel variance.

- Test 2: Misreporting and misinformation stress test
  - Vary p_overstate across 0.0, 0.2, 0.4, 0.6 to capture no misreport, baseline misreport, high misreport.
  - Metrics: same as Test 1, plus number of times you ran out of patients at a site before all planned trips completed, and resulting EV variation.
  - Goal: ensure the planning framework degrades gracefully as reports become less reliable; confirm that reallocation responds quickly to drone updates to avoid wasted trips.

- Test 3: Fairness and resource fairness test
  - Run variants where the planner is constrained to prevent repeated prioritization of a single site (e.g., impose a cap so that no site can be allocated more than X% of total “high‑priority” trips unless its patients are exhausted).
  - Metrics: distribution of waiting times for each site’s patients, maximum wait per site, Gini coefficient over wait times, EV.
  - Goal: ensure no site is systematically disadvantaged and no triage policy violates fairness constraints.

- Additional stress tests
  - Sudden, massive new information: a drone arrives with counts at all sites, showing one site has a substantial surge in need. Replan and measure improvement in EV when you reallocate mid‑stream.
  - Surge at a single site: one site has a burst of critical patients; test if the planner shifts more trips to that site while preserving overall survival.

- Metrics to report in all tests
  - Expected lives saved (EV) over 90 minutes
  - Expected number of critical survivors saved
  - Average arrival time by site and share of arrivals in each window
  - Trips completed per ambulance and unused capacity
  - Robustness metrics: standard deviation of EV across Monte Carlo runs; probability of EV falling below a threshold (e.g., <5 lives)
  - Fairness metrics: site wait time distributions, fairness index (e.g., variance of wait times across sites)

7) Ethical and legal risks and concrete mitigations

Key risks
- Risk of biased allocation if the model overfits to site sizes or relies too heavily on historical counts (which may be distorted by misreporting).
- Violation of triage ethics: even if legally permissible, automated prioritization could undermine the physician’s clinical judgment; risk of “de‑humanizing” the triage process.
- Privacy and data handling: handling patient counts and site status requires responsible data governance; ensure data is anonymized and access is restricted.
- Misreporting (deliberate misinformation) leading to suboptimal decisions and harm; overreliance on model outputs can reduce situational awareness.

Concrete mitigations
- Keep human oversight: the scheduling tool provides recommended targets but a medical commander or lead clinician must approve the final plan each step, with the ability to override automatically if safety concerns arise.
- Use universal triage rules: never use protected attributes (e.g., age, sex, race, religion) as a factor in prioritization; use objective clinical need and prognostic windows only.
- Transparency and auditability: log all decisions, inputs, and reasons for reallocations; maintain an auditable chain of decisions for post‑incident review.
- Stochastic planning with bounds: use robust optimization or stochastic programming that minimizes the worst‑case loss in expected lives saved, given uncertainty bounds about counts and delays.
- Update mechanism for misinformation: integrate real‑time drone data and reallocate accordingly. If counts drop unexpectedly, do a quick replan to avoid wasting capacity on empty sites; implement a “stop‑loss” rule to prevent overcommitting to a single site.
- Privacy safeguards: aggregate counts and de‑identify data; limit personally identifiable information exposure.

Putting it all together
- The concrete plan allocates three ambulances to perform six trips in the next 90 minutes (A1, B1, C1, A2, B2, A3) with the times and sites specified above.
- Under the model assumptions (D distribution, misreport probability 0.20), the expected lives saved is about 6.8 lives.
- If travel delays are more optimistic (Test 4a) the EV is essentially unchanged for this plan (still around 6.7–6.8). If misinformation probability rises to 0.40 (Test 4b), EV remains near the same for this plan unless counts become binding; the plan should still be robust but would merit more aggressive reallocation once drone data arrives.
- The real‑time decision algorithm is designed to adapt as events unfold (trip completions, new drone data). It uses a belief‑driven EV calculation per potential trip, replans at trip boundaries, and updates counts with drone data to avoid wasted trips.
- Robustness tests: three suggested (travel delays, misreporting, fairness) plus extra stress tests to verify that the policy performs reasonably under adverse conditions.
- Ethical/legal mitigations: maintain clinician oversight, avoid protected‑attribute prioritization, audit decisions, and ensure privacy.

If you’d like, I can tailor the numbers to reflect a specific local policy (e.g., an explicit cap on trips to each site, or a different interpretation of the “misreport by 50%” rule) or provide a simple spreadsheet/mini‑simulation script that lets you adjust p_overstate, E[D], trip capacity, and the arrival windows to see how EV changes.

In [ ]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-sonnet-4-5"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [20]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

As the autonomous coordinator for 3 ambulances after a major earthquake, my primary goal is to maximize expected lives saved over the next 90 minutes, while adhering to ethical constraints and robustness principles.

**Definitions and Initial Calculations:**

*   **Ambulance Capacity:** 3 patients per trip. Prioritize critical patients.
*   **Trip Duration:** `Total_trip_time = 2 * Actual_Travel_Time + 5 minutes (patient handling)`.
    *   `Actual_Travel_Time = Baseline_Travel_Time + Delay_D`.
*   **Delay Scenarios:**
    *   `D=0` (P=0.7)
    *   `D=10` (P=0.25)
    *   `D=20` (P=0.05)
*   **Report Scenarios (Independent per site):**
    *   `Accurate` (P=0.8)
    *   `Overstated by 50%` (P=0.2) $\rightarrow$ Actual patients = Reported * 2/3 (rounded down).
        *   Site A: (3C, 9NC) or (2C, 6NC)
        *   Site B: (6C, 14NC) or (4C, 9NC)
        *   Site C: (10C, 30NC) or (6C, 20NC)
*   **Survival Probabilities:** (Critical/Non-Critical) per site, per arrival time bin.

**Core Decision-Making Logic (Greedy Heuristic for Schedule Generation):**

To generate a concrete schedule at T=0, I will employ a greedy heuristic. At each decision point (when an ambulance becomes available), I will calculate the "expected lives saved per trip" for sending that ambulance to each available site. The ambulance will be dispatched to the site offering the highest expected value.

For the purpose of *scheduling decisions* (which trip to make next), I will use the *reported* patient counts directly (without applying misinformation probabilities) and decrement them as patients are "rescued". This simplification allows for a deterministic selection path in the greedy algorithm. The *final computation of total expected lives saved* will then rigorously account for both delay and misinformation uncertainties for each trip in the generated schedule.

**Helper Function: `get_survival_probs(site, arrival_time)`**
Returns `(survival_crit, survival_noncrit)` based on the site and arrival time bin.

**Helper Function: `pick_patients_for_trip(available_critical, available_noncritical)`**
Picks up to 3 patients, prioritizing critical patients.
Example: `(avail_C=5, avail_NC=5)` -> `(picked_C=3, picked_NC=0)`
` (avail_C=2, avail_NC=5)` -> `(picked_C=2, picked_NC=1)`
` (avail_C=0, avail_NC=2)` -> `(picked_C=0, picked_NC=2)`

**Helper Function: `calculate_trip_ELS_and_Duration(site_id, dispatch_time, current_crit_at_site, current_noncrit_at_site, delay_priors)`**
This calculates the expected lives saved and expected duration for a single trip, assuming the `current_crit_at_site` and `current_noncrit_at_site` are the *actual* available patients, and averaging over `delay_priors`.
This is used for *greedy selection* of the next trip.

**1. Concrete Allocation and Trip Schedule:**

**Initial State (T=0):**
*   Ambulances `Amb_Free = [0, 0, 0]`
*   Patient counts for scheduling (using reported counts for simplicity in dispatching):
    *   `P_Schedule = { 'A': {'C': 3, 'NC': 9}, 'B': {'C': 6, 'NC': 14}, 'C': {'C': 10, 'NC': 30} }`
*   `Schedule = []` (stores (Amb_ID, Dispatch_Time, Site, Patients_Taken_Crit, Patients_Taken_NonCrit))

**Trip 1 (Amb 1, T=0):**
*   Evaluate options:
    *   Site A (3C, 0NC): `calculate_trip_ELS_and_Duration('A', 0, 3, 9)` -> ELS: 2.085, Duration: 28 min
    *   Site B (3C, 0NC): `calculate_trip_ELS_and_Duration('B', 0, 6, 14)` -> ELS: 0.8625, Duration: 36 min
    *   Site C (3C, 0NC): `calculate_trip_ELS_and_Duration('C', 0, 10, 30)` -> ELS: 0.5775, Duration: 52 min
*   **Decision:** Amb 1 to Site A (3 critical, 0 non-critical).
*   `Schedule.append((1, 0, 'A', 3, 0))`. `Amb_Free = [28, 0, 0]`. `P_Schedule['A']['C'] = 0`.

**Trip 2 (Amb 2, T=0):**
*   Evaluate options:
    *   Site A (0C, 3NC): `calculate_trip_ELS_and_Duration('A', 0, 0, 9)` -> ELS: 2.715, Duration: 28 min
    *   Site B (3C, 0NC): `calculate_trip_ELS_and_Duration('B', 0, 6, 14)` -> ELS: 0.8625, Duration: 36 min
    *   Site C (3C, 0NC): `calculate_trip_ELS_and_Duration('C', 0, 10, 30)` -> ELS: 0.5775, Duration: 52 min
*   **Decision:** Amb 2 to Site A (0 critical, 3 non-critical).
*   `Schedule.append((2, 0, 'A', 0, 3))`. `Amb_Free = [28, 28, 0]`. `P_Schedule['A']['NC'] = 6`.

**Trip 3 (Amb 3, T=0):**
*   Evaluate options (same as Trip 2, as patient counts allow):
    *   Site A (0C, 3NC): ELS: 2.715, Duration: 28 min
    *   Site B (3C, 0NC): ELS: 0.8625, Duration: 36 min
    *   Site C (3C, 0NC): ELS: 0.5775, Duration: 52 min
*   **Decision:** Amb 3 to Site A (0 critical, 3 non-critical).
*   `Schedule.append((3, 0, 'A', 0, 3))`. `Amb_Free = [28, 28, 28]`. `P_Schedule['A']['NC'] = 3`.

**Trip 4 (Amb 1, T=28):**
*   Evaluate options:
    *   Site A (0C, 3NC): `calculate_trip_ELS_and_Duration('A', 28, 0, 3)` -> ELS: 0.6, Duration: 28 min
    *   Site B (3C, 0NC): `calculate_trip_ELS_and_Duration('B', 28, 6, 14)` -> ELS: 0.15, Duration: 36 min
    *   Site C (3C, 0NC): `calculate_trip_ELS_and_Duration('C', 28, 10, 30)` -> ELS: 0.15, Duration: 52 min
*   **Decision:** Amb 1 to Site A (0 critical, 3 non-critical).
*   `Schedule.append((1, 28, 'A', 0, 3))`. `Amb_Free = [56, 28, 28]`. `P_Schedule['A']['NC'] = 0`. (Site A now cleared)

**Trip 5 (Amb 2, T=28):**
*   Evaluate options:
    *   Site B (3C, 0NC): ELS: 0.15, Duration: 36 min
    *   Site C (3C, 0NC): ELS: 0.15, Duration: 52 min
*   **Decision:** Amb 2 to Site B (3 critical, 0 non-critical). (Tie-break: select alphabetically)
*   `Schedule.append((2, 28, 'B', 3, 0))`. `Amb_Free = [56, 64, 28]`. `P_Schedule['B']['C'] = 3`.

**Trip 6 (Amb 3, T=28):**
*   Evaluate options:
    *   Site B (3C, 0NC): ELS: 0.15, Duration: 36 min
    *   Site C (3C, 0NC): ELS: 0.15, Duration: 52 min
*   **Decision:** Amb 3 to Site C (3 critical, 0 non-critical). (Tie-break: select alphabetically for remaining)
*   `Schedule.append((3, 28, 'C', 3, 0))`. `Amb_Free = [56, 64, 80]`. `P_Schedule['C']['C'] = 7`.

**Trip 7 (Amb 1, T=56):**
*   Evaluate options:
    *   Site B (3C, 0NC): `calculate_trip_ELS_and_Duration('B', 56, 3, 14)` -> ELS: 0.15, Duration: 36 min
    *   Site C (3C, 0NC): `calculate_trip_ELS_and_Duration('C', 56, 7, 30)` -> ELS: 0.15, Duration: 52 min
*   **Decision:** Amb 1 to Site B (3 critical, 0 non-critical). (Tie-break: select alphabetically)
*   `Schedule.append((1, 56, 'B', 3, 0))`. `Amb_Free = [92, 64, 80]`. `P_Schedule['B']['C'] = 0`. (Site B critical patients cleared)

**Trip 8 (Amb 2, T=64):**
*   Evaluate options:
    *   Site B (0C, 3NC): `calculate_trip_ELS_and_Duration('B', 64, 0, 14)` -> ELS: 0.6, Duration: 36 min
    *   Site C (3C, 0NC): `calculate_trip_ELS_and_Duration('C', 64, 7, 30)` -> ELS: 0.15, Duration: 52 min
*   **Decision:** Amb 2 to Site B (0 critical, 3 non-critical).
*   `Schedule.append((2, 64, 'B', 0, 3))`. `Amb_Free = [92, 100, 80]`. `P_Schedule['B']['NC'] = 11`.

**Trip 9 (Amb 3, T=80):**
*   Evaluate options:
    *   Site B (0C, 3NC): `calculate_trip_ELS_and_Duration('B', 80, 0, 11)` -> ELS: 0.6, Duration: 36 min
    *   Site C (3C, 0NC): `calculate_trip_ELS_and_Duration('C', 80, 7, 30)` -> ELS: 0.15, Duration: 52 min
*   **Decision:** Amb 3 to Site B (0 critical, 3 non-critical).
*   `Schedule.append((3, 80, 'B', 0, 3))`. `Amb_Free = [92, 100, 116]`. `P_Schedule['B']['NC'] = 8`.

All ambulances are now expected to be available beyond the 90-minute horizon, or the trip dispatch time exceeds 90 minutes.

**Final Schedule:**
1.  **Amb 1:** Dispatch T=0, Site A, (3 Critical, 0 Non-Critical)
2.  **Amb 2:** Dispatch T=0, Site A, (0 Critical, 3 Non-Critical)
3.  **Amb 3:** Dispatch T=0, Site A, (0 Critical, 3 Non-Critical)
4.  **Amb 1:** Dispatch T=28, Site A, (0 Critical, 3 Non-Critical)
5.  **Amb 2:** Dispatch T=28, Site B, (3 Critical, 0 Non-Critical)
6.  **Amb 3:** Dispatch T=28, Site C, (3 Critical, 0 Non-Critical)
7.  **Amb 1:** Dispatch T=56, Site B, (3 Critical, 0 Non-Critical)
8.  **Amb 2:** Dispatch T=64, Site B, (0 Critical, 3 Non-Critical)
9.  **Amb 3:** Dispatch T=80, Site B, (0 Critical, 3 Non-Critical)

---

**2. Computed Expected Lives Saved (ELS) under uncertainties:**

Now, I will calculate the total ELS for the above schedule, accounting for both delay and misinformation uncertainties for each trip. This involves maintaining `Remaining_Patients_Dist` (a dictionary of `{(crit_count, noncrit_count): probability}`) for each site and updating it after each trip.

`Remaining_Patients_Dist = {`
    `'A': { (3,9): 0.8, (2,6): 0.2 },`
    `'B': { (6,14): 0.8, (4,9): 0.2 },`
    `'C': { (10,30): 0.8, (6,20): 0.2 }`
`}`
`Total_ELS = 0`

**Trip 1 (Amb 1, T=0, Site A):** ELS = 2.085. `Remaining_Patients_Dist['A'] = { (0,9): 0.8, (0,5): 0.2 }`
**Trip 2 (Amb 2, T=0, Site A):** ELS = 2.715. `Remaining_Patients_Dist['A'] = { (0,6): 0.8, (0,2): 0.2 }`
**Trip 3 (Amb 3, T=0, Site A):** ELS = 2.534. `Remaining_Patients_Dist['A'] = { (0,3): 0.8, (0,0): 0.2 }`
**Trip 4 (Amb 1, T=28, Site A):** ELS = 0.480. `Remaining_Patients_Dist['A'] = { (0,0): 1.0 }` (Site A cleared)
**Trip 5 (Amb 2, T=28, Site B):** ELS = 0.150. `Remaining_Patients_Dist['B'] = { (3,14): 0.8, (1,9): 0.2 }`
**Trip 6 (Amb 3, T=28, Site C):** ELS = 0.150. `Remaining_Patients_Dist['C'] = { (7,30): 0.8, (3,20): 0.2 }`
**Trip 7 (Amb 1, T=56, Site B):** ELS = 0.210. `Remaining_Patients_Dist['B'] = { (0,14): 0.8, (0,7): 0.2 }`
**Trip 8 (Amb 2, T=64, Site B):** ELS = 0.600. `Remaining_Patients_Dist['B'] = { (0,11): 0.8, (0,4): 0.2 }`
**Trip 9 (Amb 3, T=80, Site B):** ELS = 0.600. `Remaining_Patients_Dist['B'] = { (0,8): 0.8, (0,1): 0.2 }`

**Total Expected Lives Saved = 2.085 + 2.715 + 2.534 + 0.480 + 0.150 + 0.150 + 0.210 + 0.600 + 0.600 = 9.524**

---

**3. Clear List of Modeling Assumptions and Sensitivity:**

*   **Assumption 1: Ambulance Trip Duration Model:** `Total_trip_time = 2 * Actual_Travel_Time + 5 minutes (patient handling)`. This assumes symmetrical return travel and a fixed 5-minute loading/unloading.
    *   **Sensitivity:** This is critical. Longer handling or asymmetric return travel would reduce the number of possible trips, especially to farther sites, lowering ELS. Shorter times would allow more trips, increasing ELS.
*   **Assumption 2: Discrete Delay Distribution (D=0, 10, 20 min) with specified probabilities.**
    *   **Sensitivity:** The selected discrete points approximate a continuous distribution. If actual delays are consistently higher/lower or follow a different pattern, the ELS calculations and therefore priorities would shift. Higher delays would disproportionately penalize distant sites and reduce overall ELS.
*   **Assumption 3: Patient Counts for Misinformation Scenario are Rounded Down.** E.g., 10 critical * 2/3 = 6.66 -> 6.
    *   **Sensitivity:** Rounding down slightly reduces the expected number of patients compared to fractional values or rounding up. This can subtly influence patient picking and subsequent ELS for trips where counts are near capacity limits.
*   **Assumption 4: Greedy Heuristic for Schedule Generation.** The algorithm always picks the trip with the highest immediate ELS (using reported patient counts) for the next available ambulance.
    *   **Sensitivity:** This is a heuristic and might not yield the globally optimal schedule. A local optimum could be chosen over a better long-term strategy (e.g., clearing a farther site earlier might unlock more valuable later trips). However, it's a practical approach for real-time decision-making given computational constraints.
*   **Assumption 5: Survival Probabilities are Fixed for Time Bins and Site-Specific.**
    *   **Sensitivity:** These are given parameters. If they are inaccurate (e.g., due to unforeseen complications at a site), the ELS calculations will be flawed, potentially leading to suboptimal allocations.
*   **Assumption 6: Patient availability (critical/non-critical ratio) at a site changes dynamically with each trip (tracked via `Remaining_Patients_Dist`).**
    *   **Sensitivity:** This is a robust way to model patient depletion. The prioritization of critical patients within `pick_patients_for_trip` is a key factor. Changing this priority (e.g., equal priority, or non-critical first) would significantly alter the ELS.

---

**4. Allocation Changes under Alternate Priors:**

**Prior (a) Travel delays are more optimistic (P(D>5)=0.1, P(D>15)=0.01):**
*   **New Delay Distribution:** $P(D=0)=0.9$, $P(D=10)=0.09$, $P(D=20)=0.01$.
*   **Impact on Allocation:**
    *   **Increased Trips:** Expected trip durations significantly decrease (e.g., Site A from 28 to 23.2 min, Site B from 36 to 31.2 min, Site C from 52 to 47.2 min). More trips will be possible within 90 minutes.
    *   **Higher Survival Probabilities:** The higher probability of `D=0` means more trips arrive in better survival bins. This benefit is especially pronounced for Sites B and C, where baseline travel times often pushed them into lower survival bins under the original priors. Site C, in particular, becomes much more attractive.
    *   **Shift in Priority:** While Site A will likely still receive initial priority due to its minimal travel time, Sites B and C become much more competitive. The allocation would likely see more resources dispatched to B and C earlier in the schedule, potentially reducing the number of trips dedicated solely to non-critical patients at Site A.
*   **Expected Lives Saved (ELS):** ELS would be significantly higher across the board. For example, the first trip to Site A would yield an ELS of 2.315 (compared to 2.085). Given more trips and higher survival rates per trip, the total ELS would likely be **above 10-11**.

**Prior (b) Misinformation probability is 0.40:**
*   **New Report Distribution:** $P(accurate)=0.6$, $P(overstated by 50%)=0.4$.
*   **Impact on Allocation:**
    *   **Reduced Expected Patients:** The higher probability of misinformation leads to a lower expected number of actual patients at each site. For example, Site A's expected critical patients drop from 2.8 to 2.6, and non-critical from 8.4 to 7.8.
    *   **Higher Risk of Under-utilization:** Each dispatched ambulance has a higher probability of finding fewer patients than initially reported, leading to under-utilized capacity.
    *   **Shift in Allocation:** The overall strategy might still prioritize sites with higher reported critical counts, but the ELS per trip would be lower due to the increased uncertainty. This would make critical-heavy sites (B, C) slightly less attractive relative to A, as the "value" of each patient is discounted by the higher misinformation probability.
*   **Expected Lives Saved (ELS):** Due to the higher likelihood of finding fewer actual patients, the ELS per trip would generally decrease. While some calculations might show minor variations based on patient mix and exact rounding, the overall total ELS for the schedule would likely be **slightly lower than 9.524**, reflecting the reduced overall patient pool and increased uncertainty.

---

**5. Robust Real-Time Decision Algorithm / Pseudocode:**

**Algorithm: Dynamic Ambulance Dispatch with Belief Update**

**Data Structures:**
*   `AMBULANCES`: List of `(id, next_available_time, current_destination, current_dispatch_time, expected_patients_picked)`
*   `SITES`: Dictionary `{site_id: {base_travel_time, survival_probs, patient_distribution, historical_travel_times, current_crit_reported, current_noncrit_reported}}`
    *   `patient_distribution`: `{ (crit_count, noncrit_count): probability }` (reflects report uncertainty)
    *   `historical_travel_times`: List of `(dispatch_time, actual_travel_time)` for this site.
*   `GLOBAL_PARAMETERS`: `TIME_HORIZON=90`, `PATIENT_CAPACITY=3`, `LOADING_TIME=5`.
*   `DELAY_PRIORS`: `{0: 0.7, 10: 0.25, 20: 0.05}` (dynamically updated if data supports)
*   `REPORT_PRIORS`: `{accurate: 0.8, overstated_50: 0.2}` (dynamically updated per site if data supports)
*   `CURRENT_SIM_TIME = 0`

**Helper Function: `update_delay_priors(site_id, actual_travel_time)`**
    // Implement Bayesian update for DELAY_PRIORS based on new observations
    // For simplicity, let's assume global priors, or site-specific priors could be maintained.
    // e.g., if many trips to site A are D=0, increase P(D=0) for Site A.
    // This is a complex topic on its own. For basic pseudocode, assume fixed global priors.

**Helper Function: `calculate_trip_value(site_id, dispatch_time, site_patient_dist, delay_priors)`**
    1.  `expected_lives_saved = 0`, `expected_trip_duration = 0`
    2.  `base_tt = SITES[site_id].base_travel_time`
    3.  For `(delay_D, prob_delay)` in `delay_priors.items()`:
        *   `actual_tt = base_tt + delay_D`
        *   `arrival_time = dispatch_time + actual_tt`
        *   `survival_crit, survival_noncrit = get_survival_probs(site_id, arrival_time)`
        *   For `(patients_at_site, prob_report)` in `site_patient_dist.items()`:
            *   `num_crit, num_noncrit = patients_at_site`
            *   `picked_crit, picked_noncrit = pick_patients_for_trip(num_crit, num_noncrit)`
            *   `lives_this_subscenario = picked_crit * survival_crit + picked_noncrit * survival_noncrit`
            *   `expected_lives_saved += lives_this_subscenario * prob_delay * prob_report`
        *   `expected_trip_duration += (2 * actual_tt + GLOBAL_PARAMETERS.LOADING_TIME) * prob_delay`
    4.  Return `expected_lives_saved`, `expected_trip_duration`

**Main Loop (Real-Time Decision Algorithm):**

1.  **Initialization:**
    *   Set `CURRENT_SIM_TIME = 0`.
    *   `AMBULANCES = [(1, 0, None, None, None), (2, 0, None, None, None), (3, 0, None, None, None)]` (id, next_free_time, current_destination, dispatch_time, patients_picked_dist)
    *   Populate `SITES` with initial data, setting `patient_distribution` based on `REPORT_PRIORS`.

2.  **While `CURRENT_SIM_TIME < GLOBAL_PARAMETERS.TIME_HORIZON`:**
    *   **Find Next Event Time:** This could be an ambulance becoming free or a drone survey arriving.
        *   `next_ambulance_free_time = min(amb[1] for amb in AMBULANCES)`
        *   `next_drone_survey_time = (time of next drone data arrival, if any)`
        *   `event_time = min(next_ambulance_free_time, next_drone_survey_time)`
        *   If `event_time >= GLOBAL_PARAMETERS.TIME_HORIZON`, break.
        *   `CURRENT_SIM_TIME = event_time`.

    *   **Process Drone Survey (if at `event_time`):**
        *   Receive `drone_data = {site_id: (actual_crit_count, actual_noncrit_count)}`.
        *   For each `surveyed_site_id, (actual_crit, actual_noncrit)` in `drone_data.items()`:
            *   **Update Patient Distribution:** `SITES[surveyed_site_id].patient_distribution = { (actual_crit, actual_noncrit): 1.0 }` (Report uncertainty resolved for this site).
            *   **Re-evaluate Ongoing Trips:**
                *   For any `amb` currently en route to `surveyed_site_id` (i.e., `amb.current_destination == surveyed_site_id` AND `amb.dispatch_time + SITES[surveyed_site_id].base_travel_time > CURRENT_SIM_TIME`):
                    *   Calculate `ELS_if_continue = calculate_trip_value(surveyed_site_id, CURRENT_SIM_TIME, SITES[surveyed_site_id].patient_distribution, DELAY_PRIORS)`
                    *   (Optional, complex) Evaluate `ELS_if_divert_to_Z` for other `sites Z`. This would need re-calculating travel time from current ambulance position.
                    *   If `ELS_if_continue` is significantly lower (e.g., 0 patients found), issue `re-route` command or return to depot. This is a complex decision with associated penalties. For this problem, assume rerouting is not explicitly needed, but the *belief* about the site is updated.

    *   **Process Ambulance Availability (if at `event_time` or `next_ambulance_free_time`):**
        *   For each `amb` in `AMBULANCES` where `amb.next_free_time <= CURRENT_SIM_TIME`:
            *   **Collect Trip Data (if just finished a trip):**
                *   If `amb.current_destination` was not None:
                    *   Record `(amb.dispatch_time, actual_arrival_time, actual_patients_picked)` for `SITES[amb.current_destination].historical_travel_times`.
                    *   Use this actual arrival data to *potentially update `DELAY_PRIORS`*.
            *   **Select Next Trip:**
                *   `best_value = -1`, `chosen_site_id = None`, `chosen_duration = 0`
                *   For `site_id` in `SITES`:
                    *   `val, dur = calculate_trip_value(site_id, CURRENT_SIM_TIME, SITES[site_id].patient_distribution, DELAY_PRIORS)`
                    *   If `val > best_value`: `best_value = val`, `chosen_site_id = site_id`, `chosen_duration = dur`
                *   If `chosen_site_id` found (and `CURRENT_SIM_TIME + chosen_duration < GLOBAL_PARAMETERS.TIME_HORIZON`):
                    *   **Dispatch Ambulance:**
                        *   Update `amb.next_free_time = CURRENT_SIM_TIME + chosen_duration`.
                        *   `amb.current_destination = chosen_site_id`, `amb.dispatch_time = CURRENT_SIM_TIME`.
                        *   **Update Patient Distribution (Belief update):**
                            *   `new_patient_dist_for_site = {}`
                            *   For `(patients_at_site, prob_report)` in `SITES[chosen_site_id].patient_distribution.items()`:
                                *   `num_crit, num_noncrit = patients_at_site`
                                *   `picked_crit, picked_noncrit = pick_patients_for_trip(num_crit, num_noncrit)`
                                *   `rem_crit = num_crit - picked_crit`, `rem_noncrit = num_noncrit - picked_noncrit`
                                *   `new_patient_dist_for_site[(rem_crit, rem_noncrit)] = new_patient_dist_for_site.get((rem_crit, rem_noncrit), 0) + prob_report`
                            *   `SITES[chosen_site_id].patient_distribution = new_patient_dist_for_site`
                            *   `amb.expected_patients_picked = (picked_crit, picked_noncrit)` (for logging/evaluation)
                    *   Else (no viable trip): Set `amb.next_free_time = GLOBAL_PARAMETERS.TIME_HORIZON` (ambulance idles).

---

**6. Simulation or Stress Tests:**

1.  **Worst-Case Delay Scenario:**
    *   **Setup:** Modify `DELAY_PRIORS` to `P(D=20)=1.0` (or 0.9, making severe delays highly probable).
    *   **Metrics:** Total ELS, number of critical/non-critical patients saved, mean/median arrival time, proportion of trips arriving `>30 min`, overall ambulance utilization.
    *   **Goal:** Evaluate the algorithm's performance and adaptation under consistently poor travel conditions. It should prioritize closer sites and those with more robust survival probabilities.
2.  **High Misinformation Environment:**
    *   **Setup:** Modify `REPORT_PRIORS` to `P(overstated_50%)=0.8` (or 1.0) for all sites.
    *   **Metrics:** Total ELS, actual vs. reported patients rescued, capacity utilization (how many trips carried < 3 patients), "empty trips" (0 patients found).
    *   **Goal:** Assess the robustness to unreliable information. The algorithm should learn from drone surveys (if enabled in test) and adjust future dispatches. Expect lower overall ELS and higher ambulance "waste" if ground truth is bad.
3.  **Site-Specific Critical Failure & Drone Verification:**
    *   **Setup:**
        *   At `T=15 min`: Site C becomes inaccessible (`base_travel_time = infinity`).
        *   At `T=30 min`: A drone survey reveals Site B's actual patient count is 1 critical, 2 non-critical (much lower than reported).
    *   **Metrics:** ELS, adaptation speed (how quickly ambulances stop going to C and adjust to B's new reality), resource redistribution (how quickly other sites get more attention).
    *   **Goal:** Test adaptability to sudden environmental changes and the effectiveness of real-time belief updates from drone data.
4.  **Ambulance Breakdown:**
    *   **Setup:** Ambulance 1 breaks down permanently after completing its first trip (e.g., at T=28).
    *   **Metrics:** ELS, impact on remaining ambulances' workload, change in trip frequency/duration, overall capacity utilization with reduced fleet.
    *   **Goal:** Evaluate resilience to resource loss and how the algorithm reallocates the remaining assets.

---

**7. Ethical and Legal Risks and Mitigations:**

1.  **Risk: Prioritization Bias (ELS vs. "Worst-Off"):**
    *   **Description:** Pure ELS maximization might deprioritize small groups of critical patients at very distant sites with low survival odds, even if they are truly the "worst-off," if a trip to a closer site yields slightly more expected lives. This can lead to perceptions of unfairness or abandonment.
    *   **Mitigation:**
        *   **Minimum Threshold for Critical Care:** Implement a policy to ensure *all* critical patients, regardless of location, have at least one attempt at rescue unless their probability of survival is below a scientifically determined minimum viability threshold (e.g., 5%).
        *   **Fair Access Heuristic:** Introduce a weighted factor that slightly boosts the priority of sites that have been neglected or have a high concentration of unserved critical patients, even if their ELS is marginally lower than another option.
        *   **Human Oversight:** Flag decisions where a high-severity (critical) patient group is being bypassed for a statistically slightly better outcome elsewhere, allowing human coordinators to override based on ethical judgment.
2.  **Risk: Misinformation Amplification:**
    *   **Description:** Reliance on initial (potentially false) reports can lead to misallocation – sending ambulances to "ghost sites" or sites with fewer patients than reported, wasting resources and delaying aid to truly needy areas. The algorithm could be perceived as "stupid" for repeatedly falling for misinformation.
    *   **Mitigation:**
        *   **Multi-Source Verification:** Prioritize rapid drone/ground recon for sites with high reported numbers but low historical reliability or no independent verification. The algorithm is designed to incorporate drone updates.
        *   **Adaptive Reporting Priors:** Implement a learning mechanism where the `REPORT_PRIORS` for a specific site adjust over time based on received drone surveys or confirmed reports. If Site B consistently overstates, its `P(overstated_50%)` increases.
        *   **Cautious Initial Dispatches:** When uncertainty is high, dispatch minimum resources (e.g., one ambulance) to high-report sites initially for verification before committing the full fleet.
3.  **Risk: Algorithmic Rigidity / Local Optima:**
    *   **Description:** A greedy algorithm might get stuck in a locally optimal solution, always choosing the immediately best option, but missing a better long-term strategy (e.g., clearing a distant site early to make it available for more high-value subsequent trips).
    *   **Mitigation:**
        *   **Limited Look-Ahead:** Implement a limited search (e.g., explore 2-3 steps ahead) for particularly critical decisions or when faced with multiple options of similar ELS value.
        *   **Diversification Heuristics:** Periodically, or when ELS values are very close, introduce a small probability to "explore" alternative dispatches (e.g., sending an ambulance to a slightly lower-ELS site to gather more information or clear a different area).
        *   **Offline Scenario Testing:** Use the simulation environment (Point 6) to rigorously test the greedy algorithm against known suboptimal scenarios and refine the decision-making rules or parameters.
4.  **Risk: Legal Liability for Autonomous Decisions:**
    *   **Description:** If the autonomous system's decision results in preventable harm or death, establishing legal liability can be complex (developer, operator, AI itself).
    *   **Mitigation:**
        *   **Transparency and Explainability:** The system must be able to clearly articulate *why* it made a specific decision (e.g., "Ambulance 1 was sent to Site A because it offered an expected 2.085 lives saved, highest among available options at T=0, considering all uncertainties").
        *   **Human-in-the-Loop Oversight:** The system should serve as a decision-support tool, with a human coordinator always having the ability to review, understand, and override recommendations. All overrides must be logged with rationale.
        *   **Formal Verification & Validation:** Rigorous testing and formal methods to prove the algorithm operates as intended under defined conditions, minimizing software bugs or unintended behaviors.
        *   **Legal Frameworks:** Advocate for clear legal and regulatory frameworks governing the deployment and accountability of AI in critical life-saving operations.

In [ ]:
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
model_name = "deepseek-chat"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [21]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 3.0 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 5.6 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  11 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  17 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  21 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  26 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   2% ▕                  ▏  33 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   2% ▕                  ▏  36 MB/2.0 GB                  pulling

In [23]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(
    model=model_name, messages=messages, max_tokens=4096
)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

I can solve this problem by providing a concise and clear answer.

1. Concrete allocation and trip schedule:

Ambulance 1: Site C, critical patients ( arrival in 10 minutes)
Trip 1 + Trip 2:
- 3 critical non-critical patients from the initial count of site C, traveling back to depot at $t_A = 8$ min
 Ambulance 2: Site B, critical Non-Critical Patients (Arrival in 20 Minutes)
    - trip one:
        * Arriving three critical cases at Site B, all within the next 10 minutes and treated by 21 mins. Next trips to site b as well due to its closer location
Trip 3: Site A, non-critical patients or less severe ones (arrival in 40 min)
    - 2 patients will be picked for treating critical one at Site B arriving after 20 minutes.

2. Expected lives saved under the stated uncertainties and assumptions:

We'll use Python to compute this value with uncertainty.

```python
import numpy as np

def expected_lives_saved Site A:12, Site C :40 , Site B:20
Survival probabilities for critical patients:
Critical arrival time is within 10 minutes. Thus all could be treated.
    probability of being saved: max([0.8, 0.7])
Non-critical arrival 
Within 10 Min -> [0.95, 0.9]
Beyond 40 min-->[0.05, 0.2]   = average-0.725
Non-Critical Average survival rate = (0.95 + 0.9)/2 =  0.925

Expected Lives saved:
        # Assuming critical case treatment only in first 30 minutes after arrival.
Critical 20% chance of delays hence reducing expected survival from 0.80 to max(0.70) due
to the fact that 80% fall within time frame, while 90-95% survive.

        Average critical patients saved: ( 3 * ((1 + 0.6)-(20/30))*0.725 )=10
    
Critical patients:
Expected Critical   Lives Saved:  19*(0.70)   =13.3
Expected lives saved for site A - critical patients [from above]: 13   
Total Life saved from site A :[critical and non-critical]=31

# same calculation logic applied but with appropriate average survival rate as 
         (0.025, 0.2)
Critical Expected Average Lives Saved (Site 3):- critical   cases  = (40* (((1 + 0.6) -(5 /15)) *average survival from 10 min and above))=19 
         Total Life saved from site A : [critical and non-critical]=-38
Critical Expected Average Lives Savde: Site B ,(6 Critical )- critical cases [from above]: =12   
    Total Life Saved For site B:  [Critical + Non-Critical] =[19]
        Total lives aaved for all three sites (as calculated above)=61

```

# Modeling assumptions and sensitivity
## Travel delay assumption P(D>5)=0.3,  P(D>15)=0.05

Expected value of critical treatment if critical patient cases are treated as soon as possible after arrival i.e. before 10 minutes 
      = Max([0.8, 0.7])

Critical treatment delayed hence expected survival rate=probability for being saved - ((20/30)* chance of delayed).
    Hence probability for critical patients being saved [when delayed]=max( 0.80- (20 /30) *0.3)= max(0.7-(-20 / 30)) =0.8

    Expected Lives saved for site A: critical and non-critical = (Average Critical   Survival Rate * Total number of critical patients )
            = max([0.725, 0.925]) *12
          =12*(0.775)

         Site C expected critical patient lives saved = Max( Critical survival rate - [(((40)- 30) /15 * (P(D>5)=0.3 , P(D>15)=0.05))] 
            =max((0.8 -20/15)*0.05 + (Critical survival after the 30 min mark))=[((25-21)*0.05)+ 0.55] =11
              Total Lives Savded for  Site C   [(10)]        
             Site c Life saved [non-critical]= Max( critical rate -[((40)-35)/15*(P(D>5)=0.3)]
            = max( (9/ 15* 0.05  + 0.85) ) =(6*

Total lives aaved from site C :[(10+11), [12]]
            =   Total Lives Saved for   all three sites =[63]            
```python

from scipy.stats import norm
import numpy as np 
# assuming a normal distribution with the above- stated mean and standard deviation

def simulation (results, no_of_repetitions, interval):

    total_lives_saved=0

    lives_saved_list=[]
    results=[19 ,12  ]

    for i in range(0,no_of_repetitions):
        simulations=[] 
        initialsite_critical=Site A critical + Site B Critical + Site C Critical
        # simulation here 

        site_C_data = np.random.choice([18,26 ], p=[0.2, 0.8])
        life_saved_list_site_A site_A_critical_data =[np.random.binomial (n , average_critical_rate),
            np.random.binomial( n, critical_non_critical_averageSurvival)]
    for j in range(interval):
        
        simulation_results =[np.random.choice([ Site B +site A critical,Simulations site_B_critical ] ) ,  np.random.choice ([Simulation results Site C critical  ,
        Simulation data site c] ) ]

           lives_saved=[  n_critical_lives_saved + n_critical_site_b_data ]
```

6 Simulation Tests 
## Test-1 : Robustness when no delay is introduced

```python
#   10 min, average survival rate 0.925 

Critical expected average lives saved for SiteA [critical patients ]= Critical data from Simulation (for site A) *Average Survival Rate
site sites c critical = simulation data site c * average Critical survival in case critical patient were treated as soon as possible
        life_saved_site_C =(10+11)
       life_savedsite_A =[12*(0.78,)]

               lives saved for all siter [site A critical data  + SiteC lives saved + Sites B lives 
               saved] =[31]

Test-2: Robustness while travel delay is introduced


Critical expected average lives saved for site A critical= (critical avg survival -  (20 / 30*0.3 )
avg_critical_averagement_rate =(8/10)*0.925 *(1-(20/30) 
        average critical patient cases are to be treated by the ambulance 

Critical 7
Site critical = critical expected lifes saved for site A Critical patient data*[ 0.95 - ( (critical patients were not treated for time) 
      if (Critical data site A > Average survival time critical patients ) ]
            + (average critical patient survival rate)    
average_critical_survival_rate = [(8/10)*0.925]-((19/30)*0.3)]*0.95
            +(Critical_avg survival - (delays * average avg survival))]=12 *(0.775)

          lives saved sitec =[11]*site lives 
              life_saved_siteA critical [12*(0.78, 8*(10-20) + (average_critical_rate)   ] =(31)
         lives_saved_for all Sites [Site c  critical data  + sites Site a critical patient]
           = Site C critical and site a critical data saved

            Total Lives Saved for Site B =[19*

    sitesA critical+sitec critical data + sites B critical ] =[54]        



7 Ethicical & Legal Risks
 Mitigation  [3]: Monitoring reports with updates in real-time.

Legal Risk: 
Mitigated By Regular monitoring and updates of site C to ensure accurate figures are captured

Ethical Risk:  
Mitigating the risks by having an open, transparent system where emergency responders can report critical patient cases and information to prevent delays.
 Mitigation 2: using Drone surveys
```

In [24]:
# So where are we?

print(competitors)
print(answers)


['gpt-5-nano', 'gemini-2.5-flash', 'llama3.2', 'llama3.2']
['Below is a concrete plan and a robust accompanying analysis. I’ll keep the presentation concrete (numbers, timings, decisions) while also noting assumptions, sensitivities, and how you’d run this in real time.\n\n1) Concrete allocation and 90‑minute trip schedule (who goes where and when)\n\nAssumptions used for scheduling\n- All ambulances start at the depot at time 0.\n- Each trip is to a single site, with a return to the depot before starting another trip.\n- Each trip can treat up to 3 patients (or as many as are available at the site).\n- Travel time baseline t at site: A=8 min, B=12 min, C=20 min.\n- Independent delay D with P(D>5)=0.3 and P(D>15)=0.05 per trip; we use the expected delay E[D] ≈ 5.25 min (baseline calculation: mix of intervals 0–5, 5–15, >15; E[D] ≈ 2.5*0.7 + 10*0.25 + 20*0.05 ≈ 5.25). Each trip’s one‑way time becomes baseline + E[D].\n- Per‑trip round‑trip travel time ≈ 2 × (baseline + E[D]).\n- Reports

In [25]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5-nano

Below is a concrete plan and a robust accompanying analysis. I’ll keep the presentation concrete (numbers, timings, decisions) while also noting assumptions, sensitivities, and how you’d run this in real time.

1) Concrete allocation and 90‑minute trip schedule (who goes where and when)

Assumptions used for scheduling
- All ambulances start at the depot at time 0.
- Each trip is to a single site, with a return to the depot before starting another trip.
- Each trip can treat up to 3 patients (or as many as are available at the site).
- Travel time baseline t at site: A=8 min, B=12 min, C=20 min.
- Independent delay D with P(D>5)=0.3 and P(D>15)=0.05 per trip; we use the expected delay E[D] ≈ 5.25 min (baseline calculation: mix of intervals 0–5, 5–15, >15; E[D] ≈ 2.5*0.7 + 10*0.25 + 20*0.05 ≈ 5.25). Each trip’s one‑way time becomes baseline + E[D].
- Per‑trip round‑trip travel time ≈ 2 × (baseline + E[D]).
- Reports are uncertain: with probability 0.20 a site’s c

In [26]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [27]:
print(together)

# Response from competitor 1

Below is a concrete plan and a robust accompanying analysis. I’ll keep the presentation concrete (numbers, timings, decisions) while also noting assumptions, sensitivities, and how you’d run this in real time.

1) Concrete allocation and 90‑minute trip schedule (who goes where and when)

Assumptions used for scheduling
- All ambulances start at the depot at time 0.
- Each trip is to a single site, with a return to the depot before starting another trip.
- Each trip can treat up to 3 patients (or as many as are available at the site).
- Travel time baseline t at site: A=8 min, B=12 min, C=20 min.
- Independent delay D with P(D>5)=0.3 and P(D>15)=0.05 per trip; we use the expected delay E[D] ≈ 5.25 min (baseline calculation: mix of intervals 0–5, 5–15, >15; E[D] ≈ 2.5*0.7 + 10*0.25 + 20*0.05 ≈ 5.25). Each trip’s one‑way time becomes baseline + E[D].
- Per‑trip round‑trip travel time ≈ 2 × (baseline + E[D]).
- Reports are uncertain: with probability 0.20 a si

In [28]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [29]:
print(judge)

You are judging a competition between 4 competitors.
Each model has been given this question:

You're the autonomous coordinator of 3 ambulances after a major earthquake; you must allocate and schedule them for the next 90 minutes to maximize expected lives saved subject to legal/ethical constraints (no prioritization by protected attributes) and robustness to uncertain travel delays and possible misinformation in reports. Current situation: Site A reports 12 injured (3 critical, 9 non-critical); Site B reports 20 injured (6 critical, 14 non-critical); Site C reports 40 injured (10 critical, 30 non-critical). Survival probabilities depend on ambulance arrival time: if an ambulance arrives within 10 minutes, critical/non-critical survival probabilities are A: (0.8/0.95), B: (0.7/0.90), C: (0.6/0.85); if arrival is between 10–30 minutes they become A: (0.4/0.80), B: (0.3/0.70), C: (0.2/0.60); if arrival is after 30 minutes assume survival = 0.05 for critical and 0.20 for non‑critical. Tr

In [30]:
judge_messages = [{"role": "user", "content": judge}]

In [31]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


{"results": ["2", "1", "4", "3"]}


In [32]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gemini-2.5-flash
Rank 2: gpt-5-nano
Rank 3: llama3.2
Rank 4: llama3.2


### Extra — **Orchestrator–worker** pattern

An **orchestrator** model **plans** the work (here: split the big question into a few subtasks). **Worker** models each handle **one subtask** with a narrow prompt. The orchestrator then **merges** worker outputs into a single answer.

Run the next three cells in order (after `question` exists — same notebook as the multi-model competition).

In [33]:
# 1) Orchestrator: produce a small JSON plan of worker subtasks

orchestrator_model = "gpt-4.1-mini"

plan_prompt = f"""You are an orchestrator agent. Below is one long, multi-part question.

Split it into exactly 3 subtasks that can be answered somewhat independently by specialist workers.
Each subtask should cover a different chunk of what the question asks (for example: schedule/allocation, quantitative EV and assumptions, ethics/tests/algorithm — adapt to the actual question).

Return ONLY valid JSON (no markdown fences) in this exact shape:
{{"subtasks":[{{"id":1,"title":"short title","instructions":"what this worker must deliver"}},{{"id":2,"title":"...","instructions":"..."}},{{"id":3,"title":"...","instructions":"..."}}]}}

Full question:
---
{question}
---
"""

orch_resp = openai.chat.completions.create(
    model=orchestrator_model,
    messages=[{"role": "user", "content": plan_prompt}],
)
plan_raw = orch_resp.choices[0].message.content.strip()
if "{" in plan_raw and "}" in plan_raw:
    plan_raw = plan_raw[plan_raw.find("{") : plan_raw.rfind("}") + 1]

plan = json.loads(plan_raw)
subtasks = plan["subtasks"]
print(json.dumps(plan, indent=2))

{
  "subtasks": [
    {
      "id": 1,
      "title": "Allocation and Scheduling with Expected Lives Saved Calculation",
      "instructions": "Provide a concrete allocation and trip schedule for the 3 ambulances over the next 90 minutes, specifying which ambulance goes to which site and when. Compute the expected number of lives saved based on the given survival probabilities, travel times with uncertainty, ambulance capacity, and report inaccuracies. Clearly show how these calculations incorporate the stated uncertainties and assumptions."
    },
    {
      "id": 2,
      "title": "Scenario Analysis, Modeling Assumptions, and Impact Sensitivity",
      "instructions": "List and explain the key modeling assumptions made in the planning and calculation of ambulance allocations and expected lives saved. Analyze how sensitive the plan and expected outcomes are to each assumption. Then, describe how the ambulance allocation and expected lives saved would change under two alternate priors

In [34]:
# 2) Workers: each completes one subtask (same model = simple; you could vary per worker)

worker_model = "gpt-4.1-mini"
q_ctx = question if len(question) <= 28_000 else question[:28_000] + "\n\n[...question truncated for worker context...]"

worker_outputs = []
for st in subtasks:
    wid = int(st["id"])
    title = st["title"]
    instr = st["instructions"]
    worker_prompt = f"""You are a worker agent. Here is the full user question for context:

---
{q_ctx}
---

## Your assignment only — subtask {wid}: {title}

{instr}

Produce a focused answer for this subtask only. Use markdown headings."""

    wresp = openai.chat.completions.create(
        model=worker_model,
        messages=[{"role": "user", "content": worker_prompt}],
    )
    text = wresp.choices[0].message.content
    worker_outputs.append({"id": wid, "title": title, "content": text})
    display(Markdown(f"### Worker {wid}: {title}\n\n{text}"))

### Worker 1: Allocation and Scheduling with Expected Lives Saved Calculation

# Ambulance Allocation and Scheduling for Next 90 Minutes

---

## Summary of Key Parameters

| Site | Reported Injured | Critical (C) | Non-Critical (NC) | Baseline Travel Time (min) | Survival Probabilities (by arrival time)                        |
|-------|------------------|-------------|------------------|----------------------------|----------------------------------------------------------------|
| A     | 12               | 3           | 9                | 8                          | ≤10min: (0.8/0.95), 10-30min: (0.4/0.80), >30min: (0.05/0.20)  |
| B     | 20               | 6           | 14               | 12                         | ≤10min: (0.7/0.90), 10-30min: (0.3/0.70), >30min: (0.05/0.20)  |
| C     | 40               | 10          | 30               | 20                         | ≤10min: (0.6/0.85), 10-30min: (0.2/0.60), >30min: (0.05/0.20)  |

- Each ambulance capacity: 3 patients per trip (any mix of critical/non-critical).
- 3 ambulances, 90 minutes available.
- Random delay D for travel times:
  - P(D>5) = 0.3, P(D>15) = 0.05, independent for each trip/site.
- Report misinformation: 20% chance that reported injured counts at each site are 50% overstated (independent).
- Each trip includes going to site + return (symmetric time assumed).
- Prioritize critical patients first (no protected attributes prioritized).

---

## Step 1: Incorporate Report Uncertainty

Let \(O_s\) be indicator that site \(s\) report is overstated.

- With 80% probability: report is accurate.
- With 20% probability: actual injured = reported / 1.5.

Expected injured numbers per site:
\[
E[\text{Injured}_s] = 0.8 \times R_s + 0.2 \times \frac{R_s}{1.5}
= R_s \times (0.8 + 0.2/1.5) = R_s \times (0.8 + 0.1333) = R_s \times 0.9333
\]

Similarly for critical and non-critical:

| Site | Reported Critical | Expected Critical | Reported NonCritical | Expected NonCritical |
|-------|------------------|-------------------|---------------------|---------------------|
| A     | 3                | 3 × 0.9333 = 2.8  | 9                   | 9 × 0.9333 = 8.4    |
| B     | 6                | 5.6               | 14                  | 13.07               |
| C     | 10               | 9.33              | 30                  | 28                  |

---

## Step 2: Travel Time & Delay Distribution

Baseline travel time \(t_s\):

- Travel time per trip: \(T_s = t_s + D\),
- \(P(D \leq 5) = 0.7\),
- \(P(5 < D \leq 15) = 0.25\),
- \(P(D > 15) = 0.05\).

Use expected travel time per trip:

\[
E[D] = 0.7 \times 2.5 + 0.25 \times 10 + 0.05 \times 20 = 1.75 + 2.5 + 1 = 5.25
\]

So,

| Site | Expected Travel Time (one-way) | Round-trip (2×) Time |
|-------|-------------------------------|---------------------|
| A     | 8 + 5.25 = 13.25              | 26.5                |
| B     | 12 + 5.25 = 17.25             | 34.5                |
| C     | 20 + 5.25 = 25.25             | 50.5                |

---

## Step 3: Schedule Design Constraints

- Ambulance capacity 3 patients/trip.
- Trips include: departure from depot, travel to site, pick patients, return to depot.
- Total available duration: 90 min.
- Performance impact: Survival depends on arrival time (one-way travel from depot).

---

## Step 4: Proposed Allocation & Schedule

To **maximize expected lives saved**, preference is:

1. Reach critical patients as soon as possible at sites where survival probabilites are higher for short arrival.

2. Because site C is large but far: fewer trips possible there.

3. Sites A and B have higher survival for early arrival (especially A).

---

### Proposed Plan

| Ambulance | Trip # | Target Site | Dep Time (min) | Travel to Site LS Arrival | Expected Survival Window | #Patients Treated | Return Time (min) | Notes |
|------------|---------|-------------|----------------|--------------------------|--------------------------|-------------------|------------------|-------|
| A1         | 1       | A           | 0              | ~13.25                   | 10-30 min                | 3                 | 26.5             | First trip to closest site |
| A1         | 2       | A           | 26.5           | ~39.75                   | >30 min                  | 3                 | 53               | Second trip A |
| A1         | 3       | A           | 53             | ~66.25                   | >30 min                  | 3                 | 79.5             | Third trip A |
| A2         | 1       | B           | 0              | ~17.25                   | 10-30 min                | 3                 | 34.5             | First trip to B |
| A2         | 2       | B           | 34.5           | ~51.75                   | >30 min                  | 3                 | 68               | Second trip B |
| A2         | 3       | B           | 68             | ~85.25                   | >30 min                  | 3                 | 90+ (out of 90)  | Partial due to time |
| A3         | 1       | C           | 0              | ~25.25                   | 10-30 min+ edge          | 3                 | 50.5             | First trip to farthest site |
| A3         | 2       | C           | 50.5           | >90 arrival (too late)    | >30 min                  | 3                 | >90              | No 2nd trip feasible |

---

## Step 5: Calculate Expected Lives Saved per Site

---

### 5.1 Expected number of patients treated vs injured:

Total patients treated limited by ambulance capacity × trips:

- Site A: Ambulance A1 does 3 trips × 3 patients = 9 total patients
- Site B: Ambulance A2 does 3 trips × 3 patients but only completes 2 trips fully; third trip partial:
  - Trip 1 & 2 full (6 patients), Trip 3 partial (less than full).
- Site C: Ambulance A3 does 1 trip (3 patients), second trip not feasible within 90 min.



### 5.2 Patient priority: Critical patients first

| Site | Expected Critical | Expected Noncritical | Total Expected Injured | Treated Capacity | Critical Treated | Noncritical Treated |
|-------|------------------|---------------------|-----------------------|------------------|-----------------|--------------------|
| A     | 2.8              | 8.4                 | 11.2                  | 9                | 2.8 (all)       | 6.2                |
| B     | 5.6              | 13.07               | 18.67                 | 8.5 approx (2.8 trips ×3)         | 5.6 (all)       | 2.9                |
| C     | 9.33             | 28                  | 37.3                  | 3                | 3               | 0                  |

---

### 5.3 Survival probabilities per patient treated at each site and trip

#### Site A:

All 3 trips arrive at:

- Trip 1: arrival ~13.25 min → survival window 10-30 min  
  - crit: 0.4, noncrit: 0.8

- Trip 2 & 3: arrival >30 min  
  - crit: 0.05, noncrit: 0.20

Apply for expected treated patient groups:

- Critical patients (2.8 treated) all on trip 1 and 2 (assume trip 1 covers 2.8 critical + part noncritical)

- Noncritical (6.2 treated): trip 1 and 2 split; let's allocate critical patients first on trip 1 (capacity 3), trip 1 treats 2.8 critical + 0.2 noncritical; trip 2 treats the rest: 6 noncritical

Therefore:

- Trip 1: 2.8 critical × 0.4 survival + 0.2 noncritical × 0.8 survival  
- Trip 2 & 3: 0.2 noncritical × 0.05 + 6 noncritical × 0.20 survival (split between trips)

Approximate treated noncritical on trip 2 & 3 = 6

Expected lives saved trip 1:

\[
2.8 \times 0.4 + 0.2 \times 0.8 = 1.12 + 0.16 = 1.28
\]

Expected lives saved trip 2+3 (noncritical):

\[
6 \times 0.20 = 1.2
\]

Total Site A expected saved = \(1.28 + 1.2 = 2.48\)

---

#### Site B:

Travel time ~17.25 min first trip → 10-30 min survival window

- critical survival = 0.3, noncritical = 0.7 for trips within 10–30 min
- trips 2–3 arrival >30 min → 0.05/0.20 survival

Trips 1 and 2 are within 90 min; trip 3 just barely feasible arrival time ~85 min (out of scope for fast survival)

Capacity treated patients ≈ 8.5 (2 full trips + partial 3rd)

Critical treated: all 5.6 critical patients first.

Noncritical treated: (8.5 - 5.6) = 2.9 noncritical.

Assign:

- Trip 1: 3 critical → survival 0.3
- Trip 2: 2.6 critical + 0.4 noncritical (to complete critical) → survival 0.3 & 0.7
- Trip 3: 2.5 noncritical → survival 0.20 (arrival >30)

Expected lives from trip 1 (critical):

\[
3 \times 0.3 = 0.9
\]

Trip 2 (critical + noncritical):

\[
2.6 \times 0.3 + 0.4 \times 0.7 = 0.78 + 0.28 = 1.06
\]

Trip 3 (noncritical):

\[
2.5 \times 0.20 = 0.5
\]

Total expected lives Site B:

\[
0.9 + 1.06 + 0.5 = 2.46
\]

---

#### Site C:

First trip arrival time ~25.25 min (10-30 min window):

- critical survival = 0.2
- noncritical = 0.6

Capacity treated: 3 patients (assume critical first)

Critical treated: 3 (out of 9.33 critical)

Expected lives saved for trip 1:

\[
3 \times 0.2 = 0.6
\]

No further trips feasible.

---

## Step 6: Total Expected Lives Saved

Sum over sites:

\[
2.48 + 2.46 + 0.6 = 5.54 \text{ expected lives saved}
\]

---

# **Final Allocation Summary**

| Ambulance | Trips | Site  | Trip Time Frames (Departure-Return, min)              | Patients Treated                      | Approx. Expected Lives Saved |
|------------|-------|-------|-------------------------------------------------------|-------------------------------------|------------------------------|
| A1         | 3     | A     | [0–26.5], [26.5–53], [53–79.5]                        | 9 total (2.8 crit, 6.2 noncrit)     | 2.48                         |
| A2         | 3 (partial last)     | B     | [0–34.5], [34.5–68], [68–90+]               | ~8.5 total (5.6 crit, 2.9 noncrit)  | 2.46                         |
| A3         | 1     | C     | [0–50.5]                                              | 3 crit                              | 0.6                          |

---

# Notes on Uncertainty Incorporation

- Report inaccuracies reduce expected injured counts by factor 0.9333.
- Travel delay included in expected travel times and reflected in arrival classification.
- Survival probabilities adjusted by travel time windows.
- Partial third trips allocated only when possible within 90 minutes.
- Treatment priority: critical first, then noncritical.

---

# Summary

- Assign Ambulance 1 to Site A (closest, medium severity),
- Ambulance 2 to Site B (next priority),
- Ambulance 3 to Site C (largest injured count but longest travel).
- Perform multiple trips as time allows.
- Total expected lives saved under travel and misinformation uncertainty: **~5.54 lives**.

---

# End of subtask 1.

### Worker 2: Scenario Analysis, Modeling Assumptions, and Impact Sensitivity

# Subtask 2: Scenario Analysis, Modeling Assumptions, and Impact Sensitivity

---

## Key Modeling Assumptions

1. **Survival Probability Tiers by Arrival Time**  
   - Survival probabilities are fixed and stepwise depending on ambulance arrival intervals (<10 min, 10–30 min, >30 min).  
   - Assumes a deterministic drop in survival based strictly on arrival time brackets, ignoring continuous or nonlinear survival decay.

2. **Ambulance Capacity and Multiple Trips**  
   - Each ambulance carries up to 3 patients per trip and can perform multiple back-and-forth trips sequentially within 90 minutes.  
   - Assumes negligible loading/unloading time beyond travel times.

3. **Travel Times and Delay Distribution**  
   - Baseline travel times: A=8, B=12, C=20 minutes from depot.  
   - Independent travel delays D with probabilities: P(D > 5 min) =0.3 and P(D >15 min)=0.05, delays added on top of baseline.  
   - Assumes delays are independent across ambulances/trips and sites.

4. **Report Count Uncertainty (Misinformation Probability)**  
   - Each site’s reported injured count may be overstated by 50% independently with probability 0.20.  
   - Assumes no understating and that inflation—when it occurs—is exactly 50%.

5. **Homogenous Patient Prioritization (No Protected Attribute Bias)**  
   - Allocation and scheduling only depend on patient criticality and expected survival benefits, ignoring any protected attribute differences.

6. **Independence Between Sites**  
   - Delays and misinformation probabilities are independent events site-to-site.  
   - No spatial or temporal correlation (e.g., delays at one site don’t affect another).

7. **Limited Planning Horizon of 90 Minutes**  
   - Ambulance allocation and trips are optimized only within 90 minutes, ignoring longer-term strategies beyond this window.

---

## Sensitivity Analysis of Key Assumptions

| Assumption | Sensitivity and Impact on Plan & Outcomes |
|------------|-------------------------------------------|
| **Survival probabilities by arrival bracket** | Moderate: Step-function survival simplifies optimization but masks nuances. Small timing misestimates can disproportionately impact expected lives saved, especially near thresholds (e.g., 10 or 30 minutes). |
| **Ambulance capacity and multiple trips** | High: Assumes no delay loading/unloading; if these times are significant, fewer trips possible, reducing coverage—plan highly relies on quick turnaround. |
| **Travel delay probabilities** | High: Stochastic delays greatly affect arrival times and survival probabilities; overestimation or underestimation leads to misallocation (e.g., sending resources to distant sites with underestimated delays). |
| **Report misinformation probability** | Moderate to High: Overstated injury counts lead to over-resourcing some sites, potentially neglecting others, hurting overall lives saved. Independent inflation assumption affects risk diversification; unmodeled underreporting could reverse results. |
| **No patient bias** | Low in expected lives saved but crucial ethically. Violations here could skew allocation unfairly. |
| **Independence assumptions** | Moderate: Correlated delays or misinformation could lead to systemic biases or failures, not captured here, reducing robustness. |
| **90-minute horizon** | Low to Moderate: Short horizon focuses efforts but may underutilize ambulances on longer trips; changing horizon may shift allocation priorities. |

---

## Changes Under Alternate Priors

### (a) More Optimistic Travel Delay Probabilities  
- Revised: P(D > 5) = 0.1; P(D > 15) = 0.01

**Impact:**  
- Expected travel delays and variance drop significantly. Ambulances have more reliable and shorter arrival times.  
- Higher confidence in arrival times <10 or 10–30 minutes, improving survival probabilities.  
- Makes serving distant Site C more attractive since delay risk is less.  
- Enables more trips overall within 90 minutes due to predictable timings.

**Expected Lives Saved Estimate:**  
Recomputing with lower delay probabilities increases survival probabilities weighted by arrival times and hence lives saved. Assuming linear approximate increase:

- Original expected lives saved: ~[computed in initial task; let's say approx. 42 lives for baseline]  
- With delays reduced ~30% less likely >5 min and >15 min, survival achievements for critical and non-critical patients improve roughly by ~10–15%.  
- **New expected lives saved ≈ 46–48 lives.**

**Likely Allocation Changes:**  
- More trips to Site C and Site B due to improved transit reliability.  
- Possibly allocate two ambulances to Site C trips sequentially, one to Site A, enabling more patients to be transported quickly.

---

### (b) Increased Misinformation Probability (0.40 vs 0.20)  
- Revised: Probability of 50% overstatement per site increases to 40%.

**Impact:**  
- Greater uncertainty and risk that actual injured counts are less than reported.  
- Could cause over-allocation to sites with inflated counts, leaving fewer resources for others.  
- Reduces expected lives saved due to misallocation.

**Expected Lives Saved Estimate:**  
- Given increased misinformation, expected patients actually needing care decrease but ambulance allocation remains same, creating inefficiency.  
- The expected lives saved could drop by approximately 10–15% from baseline due to over- or under-utilization risks.  
- **New expected lives saved ≈ 35–38 lives.**

**Likely Allocation Changes:**  
- More conservative allocation, possibly splitting ambulances more evenly across sites instead of heavily favoring one, to hedge against misinformation.  
- Scheduling includes standby intervals to adapt upon receiving updated information.

---

# Summary Table of Expected Lives Saved Estimates

| Scenario                     | Expected Lives Saved | Primary Allocation Impact                           |
|------------------------------|---------------------|----------------------------------------------------|
| Baseline (P(D>5)=0.3, P(misinform)=0.2) | ~42                  | Balanced trips favoring critical need and proximity |
| (a) Optimistic Delay (P(D>5)=0.1, P(misinform)=0.2)   | ~46–48               | More trips to farther sites, improved timing       |
| (b) Increased Misinformation (P(D>5)=0.3, P(misinform)=0.4) | ~35–38               | More cautious, risk hedging allocations             |

---

# Conclusion

The plan is highly sensitive to **travel delay assumptions**, which dominate arrival time uncertainty and survival outcomes. **Misinformation uncertainty** critically affects allocation efficiency but less so the physical capabilities of ambulances. Adjusting priorities and schedules dynamically based on updated belief states (e.g., from drone surveys) can mitigate misinformation impacts to a degree.

### Worker 3: Robust Decision Algorithm with Fairness and Ethical Considerations

# Robust Real-Time Decision Algorithm with Fairness and Ethical Considerations

---

## 1. Robust Real-Time Decision Algorithm (Pseudocode)

```python
# Initialize belief state for each site
# Each site: belief over injury counts = mixture of reported count and 50% overstated count with known probabilities
# Example: P(report accurate) = 0.8, P(overstated) = 0.2
# Survival probabilities and travel time distributions are known a priori

initialize_beliefs(site_reports):
    beliefs = {}
    for site, report in site_reports.items():
        # Belief is a distribution over actual injured counts
        beliefs[site] = {
            "prob_accurate": 0.8,
            "injured_counts": {
                "critical": report['critical'],
                "non_critical": report['non_critical']
            },
            "prob_overstated": 0.2,
            "injured_counts_overstated": {
                "critical": int(report['critical'] * 1.5),
                "non_critical": int(report['non_critical'] * 1.5)
            }
        }
    return beliefs

# Update beliefs based on new drone survey data
update_beliefs(beliefs, drone_survey_data):
    for site, data in drone_survey_data.items():
        # Drone data considered more reliable, update beliefs using Bayesian update or weighted average
        # Here assume drone updates actual counts; adjust probabilities accordingly
        if data:
            # Reduce misinformation probability or set beliefs close to drone counts
            beliefs[site]['prob_accurate'] = 0.95
            beliefs[site]['prob_overstated'] = 0.05
            beliefs[site]['injured_counts'] = data
            beliefs[site]['injured_counts_overstated'] = {
                "critical": int(data['critical'] * 1.5),
                "non_critical": int(data['non_critical'] * 1.5)
            }
    return beliefs

# Sample travel delay for each trip based on delay distribution
sample_travel_delay():
    # Randomly sample delay D with given probabilities
    rand_val = random.uniform(0,1)
    if rand_val <= 0.05:
        return 15 + random_exp() # random_exp() some small added time
    elif rand_val <= 0.35:
        return 5 + random_exp()
    else:
        return 0

# Compute expected survival for a site given ambulance arrival times and loads
compute_expected_survival(beliefs, arrival_times):
    # For each site, weighted survival probabilities based on belief of true counts
    total_expected_survival = 0
    for site, belief in beliefs.items():
        # For simplicity, consider expected counts as weighted average
        expected_critical = belief['prob_accurate'] * belief['injured_counts']['critical'] + \
                           belief['prob_overstated'] * belief['injured_counts_overstated']['critical']

        expected_non_critical = belief['prob_accurate'] * belief['injured_counts']['non_critical'] + \
                               belief['prob_overstated'] * belief['injured_counts_overstated']['non_critical']

        arrival_time = arrival_times[site]  # in minutes

        # Determine survival probabilities per time bucket
        if arrival_time <= 10:
            surv_crit = survival_probs[site]['critical'][0]
            surv_noncrit = survival_probs[site]['non_critical'][0]
        elif arrival_time <= 30:
            surv_crit = survival_probs[site]['critical'][1]
            surv_noncrit = survival_probs[site]['non_critical'][1]
        else:
            surv_crit = survival_probs[site]['critical'][2]
            surv_noncrit = survival_probs[site]['non_critical'][2]

        total_expected_survival += expected_critical * surv_crit + expected_non_critical * surv_noncrit

    return total_expected_survival

# Core decision loop to allocate ambulances dynamically
real_time_scheduler(beliefs, ambulances, time_remaining):
    schedule = []
    while time_remaining > 0 and ambulances:
        # For each ambulance, select the trip (site and patient load) maximizing marginal expected survival gain
        best_allocation = None
        best_gain = -inf

        for ambulance in ambulances:
            for site in beliefs:
                # Estimate travel time with delay sampling
                travel_time = baseline_times[site] + sample_travel_delay()

                # Estimate arrival time = current time + travel_time
                arrival_time = current_time + travel_time

                # Compute marginal expected survival gain if ambulance dispatched here now
                # Note: consider ambulance capacity (3 patients), prioritize critical patients first (ethically neutral since same across sites)

                marginal_gain = estimate_marginal_gain(beliefs[site], arrival_time, capacity=3)

                if marginal_gain > best_gain:
                    best_gain = marginal_gain
                    best_allocation = (ambulance, site, travel_time)

        if best_allocation is None:
            break

        ambulance, site, travel_time = best_allocation
        # Schedule the trip: dispatch ambulance to site
        schedule.append({
            'ambulance': ambulance,
            'site': site,
            'depart_time': current_time,
            'arrival_time': current_time + travel_time,
            'patients_transported': min(3, ...)
        })

        # Update beliefs by reducing injured counts at the site accordingly
        update_beliefs_post_trip(beliefs, site, patients_treated=3)

        # Update ambulance availability considering round-trip time (travel_time * 2 + treatment time assumed negligible or fixed)
        ambulances[ambulance] = current_time + 2*travel_time + treatment_buffer

        # Update current time or time_remaining accordingly
        time_remaining -= travel_time + treatment_buffer  # conservative estimate

    return schedule

```

---

## 2. Simulation / Stress Tests and Metrics

### Stress Test 1: **Extreme Travel Delays**

- **Setup:** Simulate multiple scenarios where travel delays frequently exceed 15 minutes.
- **Purpose:** Test robustness of schedule when travel times are heavily impacted.
- **Metrics:**
  - Expected lives saved versus baseline.
  - Distribution of arrival times.
  - Ambulance utilization efficiency.

### Stress Test 2: **High Misinformation Rates**

- **Setup:** Increase misinformation probability from 20% up to 50% in simulation.
- **Purpose:** Evaluate algorithm’s ability to adapt beliefs and resource allocation amid false reports.
- **Metrics:**
  - Difference in expected vs. actual lives saved.
  - Frequency and magnitude of belief updates post-drone surveys.
  - Fairness metrics: measure equitable survival chances across sites (e.g., survival rate variance).

### Stress Test 3: **Protected Attribute Neutrality**

- **Setup:** Incorporate synthetic demographic labels while ensuring no prioritization based on them.
- **Purpose:** Validate fairness by confirming no systematic bias in ambulance assignment related to protected attributes (e.g., age, ethnicity).
- **Metrics:**
  - Statistical parity difference in ambulance response times across groups.
  - Treatment equality: compare survival improvements across groups.
  - Disparate impact ratio.

---

## 3. Legal and Ethical Risks and Mitigations

| Risk                                                 | Mitigation                                                                                  |
|------------------------------------------------------|---------------------------------------------------------------------------------------------|
| **Unintentional bias in allocation violating equality**    | Algorithm explicitly excludes protected attributes; audits performed regularly.            |
| **Inaccurate beliefs leading to suboptimal or unfair allocations** | Incorporate Bayesian updates from reliable external sources (drone surveys); conservative estimations when uncertain. |
| **Over/under triage due to misinformation or delays**       | Maintain flexible reallocation policies; continuous monitoring and real-time update capabilities. |
| **Privacy concerns with data from drone surveys**            | Use anonymized, aggregated data only; comply with data protection regulations (e.g., HIPAA). |
| **Failure to meet legal obligations for equal access to treatment** | Transparent decision criteria; documentation of decision-making process; human oversight to intervene. |

---

# Summary

- **Algorithm** dynamically updates beliefs on injury counts using reported data and drone survey updates, recalculates schedules optimizing expected lives saved under travel delays and misinformation uncertainties.
- **Stress tests** evaluate resilience to delays, misinformation, and fairness against protected attributes.
- **Risks** include potential bias and misinformation impacts; mitigated via exclusion of protected attributes, Bayesian updates, human oversight, and privacy measures.

In [36]:
# 3) Orchestrator merge: stitch worker outputs into one coherent response

bundle = "\n\n---\n\n".join(
    f"## Worker {w['id']}: {w['title']}\n\n{w['content']}" for w in sorted(worker_outputs, key=lambda x: x["id"])
)
q_head = question[:6_000] + ("\n\n[...]" if len(question) > 6_000 else "")

merge_prompt = f"""You are the orchestrator again. Workers answered parts of the same hard question.

Reminder of the question (start):
---
{q_head}
---

Worker outputs:
---
{bundle[:72_000]}{"...[bundle truncated]" if len(bundle) > 72_000 else ""}
---

Write ONE integrated answer for the user: merge the worker material, remove repetition, and align with the original question's numbered parts where possible. Use clear markdown structure."""

merge_resp = openai.chat.completions.create(
    model=orchestrator_model,
    messages=[{"role": "user", "content": merge_prompt}],
)
display(Markdown("## Orchestrator merged answer\n\n" + merge_resp.choices[0].message.content))

## Orchestrator merged answer

Certainly! Below is a comprehensive, integrated answer to your complex ambulance allocation and scheduling problem following the original question's structure. It merges the workers’ outputs into a coherent, concise, and clear response.

---

# Ambulance Allocation and Scheduling to Maximize Expected Lives Saved  
### After Major Earthquake: Sites A, B, C — 90-Minute Horizon

---

## 1) Concrete Allocation & Trip Schedule

### Situation Summary

| Site | Reported Injured (Critical / Non-Critical) | Baseline Travel Time (min, one-way) | Survival Probability by Arrival Time (Critical / Non-Critical) |
|-------|-------------------------------|-----------------------|-------------------------------------------------------------------|
| A     | 3 / 9 = 12 total              | 8                     | ≤10min: 0.8 / 0.95; 10–30min: 0.4 / 0.80; >30min: 0.05 / 0.20   |
| B     | 6 / 14 = 20 total             | 12                    | ≤10min: 0.7 / 0.90; 10–30min: 0.3 / 0.70; >30min: 0.05 / 0.20   |
| C     | 10 / 30 = 40 total            | 20                    | ≤10min: 0.6 / 0.85; 10–30min: 0.2 / 0.60; >30min: 0.05 / 0.20   |

- Ambulances: 3 total  
- Capacity: 3 patients per trip (any critical/non-critical mix)  
- Planning horizon: 90 minutes  
- Travel delay D: random, independent per trip/site, with P(D>5)=0.3 and P(D>15)=0.05  
- Report uncertainty: 20% chance site counts overstated by 50%, independently per site   
- Prioritization: critical patients first, no bias on protected attributes  
- Trips include round-trip time (site + return to depot)  
- Assumed negligible loading/unloading time compared to travel  

---

### Incorporating Report Uncertainty

Actual injured expected numbers:

\[
E[\text{injured}] = 0.8 \times R + 0.2 \times \frac{R}{1.5} = 0.9333 \times R
\]

Expected critical / non-critical injured:

| Site | Expected Critical | Expected Non-Critical |
|-------|------------------|----------------------|
| A     | 2.8              | 8.4                  |
| B     | 5.6              | 13.1                 |
| C     | 9.3              | 28                   |

---

### Incorporating Travel Delay Expectation

Expected delay:

\[
E[D] = 0.7 \times 2.5 + 0.25 \times 10 + 0.05 \times 20 = 5.25 \text{ min}
\]

Expected one-way travel times:

| Site | Baseline \(t_s\) | Expected One-Way Time \(t_s + E[D]\) | Round-Trip (2×) Time |
|-------|-----------------|-------------------------------------|----------------------|
| A     | 8               | 13.25                              | 26.5                 |
| B     | 12              | 17.25                              | 34.5                 |
| C     | 20              | 25.25                              | 50.5                 |

---

### Scheduling Plan (Fixed Allocation)

| Ambulance | Number of Trips | Site | Departure / Return Time Windows (min) | Patients Treated (Capacity 3/trip) | Notes             |
|-----------|-----------------|------|----------------------------------------|-----------------------------------|-------------------|
| A1        | 3               | A    | [0–26.5], [26.5–53], [53–79.5]        | 9 total (all critical + non-critical)       | Closest site, multiple trips |
| A2        | 3 (last partial)| B    | [0–34.5], [34.5–68], [68–90+]          | ~8.5 treated patients (full two trips + partial third) | Moderate travel time|
| A3        | 1               | C    | [0–50.5]                              | 3 critical only                  | Long travel time limits trips |

- Critical patients prioritized first per site and trip.
- Arrival times put Trips 1 (for A and B) mostly in 10–30 min survival window.
- Later trips exceed 30 min windows, reduced survival prospects.

---

## 2) Computed Expected Lives Saved Under Uncertainties

**Site A:**

- Critical treated: 2.8 (all in trip 1, arrival ~13.25 min → 10–30 min window → survival 0.4)  
- Non-critical treated: 6.2 spread over trips 1–3  
- Expected lives saved:  
  - Trip 1 critical: \(2.8 \times 0.4 = 1.12\)  
  - Trip 1 non-critical (approx 0.2 pts): \(0.2 \times 0.8 = 0.16\)  
  - Trips 2 & 3 non-critical (6 pts): \(6 \times 0.20 = 1.2\)  
  - **Total: ~2.48 lives**

---

**Site B:**

- Critical treated: 5.6 (split trip 1 and 2, 10–30 min window, survival 0.3)  
- Non-critical treated: 2.9 (trip 2 and 3, last >30 min, survival 0.20)  
- Expected lives saved:  
  - Trip 1 critical (3 pts): \(3 \times 0.3 = 0.9\)  
  - Trip 2 critical + non-critical (approx 3 pts): \((2.6 \times 0.3) + (0.4 \times 0.7) = 0.78 + 0.28 = 1.06\)  
  - Trip 3 non-critical (2.5 pts): \(2.5 \times 0.20 = 0.5\)  
  - **Total: ~2.46 lives**

---

**Site C:**

- Only 1 trip possible (3 critical pts), arrival ~25.25 min (10–30 min window), survival 0.2  
- Expected lives saved: \(3 \times 0.2 = 0.6\)

---

### Total Expected Lives Saved = 2.48 + 2.46 + 0.6 = **~5.54 lives**  

---

## 3) Modeling Assumptions and Sensitivity

| Assumption                     | Description & Impact                                                                       |
|-------------------------------|------------------------------------------------------------------------------------------|
| Survival Rates as Step Functions | Survival probabilities depend on discrete time buckets only; arrival time classification critical, small timing errors impact estimates significantly. |
| Ambulance Capacity and Trips   | 3 patients/trip, multiple sequential trips possible, loading/unloading times negligible — if not, fewer trips leading to lower coverage. |
| Travel Delay Model             | Independent random delay per trip/site with fixed probabilities; impacts arrival classification and thus survival; dominates uncertainty. |
| Report Misinformation          | 20% chance that counts overstated by 50%; no understating modeled; influences expected patient counts and resource allocation. |
| Independence Across Sites      | Delays and misinformation events considered independent, ignoring spatial/temporal correlations; correlated events may reduce robustness. |
| Priority solely by Criticality | No protected attribute considered or biased; ethical/legal compliance enforced; prioritization impacts allocation effectiveness. |
| 90-Minute Planning Horizon     | Ignores longer-term scheduling and reallocation beyond 90 minutes; limits number of trips particularly to distant sites. |

**Sensitivity:**

- **Travel delays:** High sensitivity; inaccurate delay distributions lead to misestimated survival and resource wastage.
- **Reporting misinformation:** Moderate to high sensitivity; inaccurate counts cause suboptimal ambulance allocation.
- **Survival time buckets:** Moderate; stepwise approximation ignores gradual survival changes.
- **Capacity assumptions:** High; unaccounted loading delays reduce trips and lives saved.
- **Independence assumptions:** Moderate; ignoring correlations can reduce robustness.

---

## 4) Allocation & Expected Lives Saved Under Alternate Priors

| Scenario                             | Changes to Priors                             | Impact on Allocation                             | Expected Lives Saved Estimate          |
|------------------------------------|----------------------------------------------|-------------------------------------------------|---------------------------------------|
| (a) More Optimistic Travel Delays  | P(D>5) = 0.1, P(D>15) = 0.01 (down from 0.3/0.05) | Travel times shorter and more predictable; ambulance trips to farther Site C more feasible in 90 min; possibly 2 ambulances to Site C, 1 to A. | Increased survival due to faster arrivals; expected lives saved increase ~10–15% → Estimated **~6.1 lives** (proportional to baseline 5.54). |
| (b) Increased Misinformation (40%) | Probability that reports overstated by 50% doubled to 40% | Higher uncertainty causes cautious or more even ambulance splitting; avoids over-committing to possibly overstated sites. | More misallocated trips and inefficient coverage; expected lives saved decrease ~10–15% → Estimated **~4.7 lives**.|

---

## 5) Robust Real-Time Decision Algorithm (Pseudocode)

```python
# Initial belief state for sites: mixture model for injury counts with probabilities of accurate vs overstated reports
beliefs = initialize_beliefs(site_reports)  # e.g., 80% accurate, 20% overstated by 50%

def update_beliefs(beliefs, drone_data):
    """
    Incorporate external drone survey to update beliefs on actual injury counts,
    reducing misinformation uncertainty.
    """
    for site, data in drone_data.items():
        if data is not None:
            beliefs[site]['prob_accurate'] = 0.95
            beliefs[site]['prob_overstated'] = 0.05
            beliefs[site]['counts'] = data
    return beliefs

def sample_travel_delay():
    """
    Sample random travel delay for trip using known delay distribution.
    """
    p = random.random()
    if p < 0.05:
        return uniform(15, 20)
    elif p < 0.35:
        return uniform(5, 15)
    else:
        return uniform(0, 5)

def estimate_marginal_gain(belief, arrival_time, capacity=3):
    """
    Compute expected marginal lives saved by dispatching ambulance
    based on belief about injured counts, survival probabilities for arrival_time,
    and capacity.
    """
    expected_critical = belief['prob_accurate'] * belief['counts']['critical'] + \
                        belief['prob_overstated'] * belief['counts_overstated']['critical']
    expected_noncritical = belief['prob_accurate'] * belief['counts']['non_critical'] + \
                           belief['prob_overstated'] * belief['counts_overstated']['non_critical']

    # Prioritize critical patients
    crit_treated = min(expected_critical, capacity)
    noncrit_treated = min(expected_noncritical, capacity - crit_treated)

    # Determine survival probabilities bucket
    if arrival_time <= 10:
        surv_crit, surv_noncrit = survival_probs[site]['critical'][0], survival_probs[site]['non_critical'][0]
    elif arrival_time <= 30:
        surv_crit, surv_noncrit = survival_probs[site]['critical'][1], survival_probs[site]['non_critical'][1]
    else:
        surv_crit, surv_noncrit = survival_probs[site]['critical'][2], survival_probs[site]['non_critical'][2]

    return crit_treated * surv_crit + noncrit_treated * surv_noncrit

def real_time_scheduler(beliefs, ambulances, current_time, time_limit=90):
    """
    Dynamic scheduling: iteratively assign ambulances to trips that maximize expected lives saved.
    Update beliefs and ambulance availability after each trip.
    Incorporates travel delay sampling.
    """
    schedule = []
    while current_time < time_limit and ambulances:
        best_gain = -float('inf')
        best_action = None
        for amb in ambulances:
            for site in beliefs:
                baseline_tt = baseline_times[site]
                delay = sample_travel_delay()
                travel_time = baseline_tt + delay
                arrival = current_time + travel_time
                if arrival > time_limit:
                    continue
                gain = estimate_marginal_gain(beliefs[site], arrival)
                if gain > best_gain:
                    best_gain = gain
                    best_action = (amb, site, current_time, travel_time)
        if best_action is None:
            break

        amb, site, depart_time, travel_time = best_action

        # Schedule trip
        schedule.append({
            'ambulance': amb,
            'site': site,
            'depart': depart_time,
            'arrival': depart_time + travel_time,
            'return': depart_time + 2*travel_time,
            'patients': 3
        })

        # Update belief post-trip - reduce treated injured counts
        reduce_counts(beliefs[site], patients=3)

        # Ambulance next availability
        ambulances[amb] = depart_time + 2*travel_time  

        current_time = depart_time  # or advance time to earliest available ambulance if parallel scheduling

    return schedule
```

- Bayesian belief update with drone surveys reduces misinformation uncertainty.
- Ambulances assigned dynamically to sites with highest marginal expected lives saved.
- Real-time incorporation of travel delay realizations.
- Patient priorities respected without bias.
- Reallocation possible upon new info to increase robustness.

---

## 6) Simulation / Stress Tests and Metrics

| Stress Test               | Description                                    | Metrics to Report                            |
|--------------------------|------------------------------------------------|---------------------------------------------|
| 1. Extreme Travel Delays  | Frequent >15 min delays; test scheduling resilience.  | Expected lives saved; arrival time distributions; ambulance utilization efficiency. |
| 2. High Misinformation Rate | Test with misinformation probability up to 50%.   | Expected vs actual lives saved; belief update frequency and accuracy; allocation stability. |
| 3. Protected Attribute Neutrality | Simulate demographics ensuring no bias.         | Statistical parity of ambulance arrival times; survival outcomes equity; disparate impact ratio. |

---

## 7) Ethical and Legal Risks with Mitigations

| Risk                                                        | Mitigation                                                                                          |
|-------------------------------------------------------------|---------------------------------------------------------------------------------------------------|
| Bias or discrimination based on protected attributes        | Explicit exclusion of protected attributes from algorithm inputs; regular fairness audits.         |
| Resource misallocation due to inaccurate or biased reports  | Incorporate Bayesian updates with reliable data like drone surveys; maintain adaptable schedules. |
| Privacy concerns from data sources (e.g., drones)            | Ensure data anonymization; comply with relevant laws (HIPAA or GDPR equivalents).                  |
| Over-triage or under-triage due to uncertainty               | Maintain conservative planning with dynamic reallocation; human oversight in decision loops.     |
| Lack of transparency and explainability                      | Document algorithm logic and decisions; provide explainable intent and logs.                      |
| Failure to meet legal requirements for equal access         | Transparent criteria and legal compliance checks; human audit and intervention capability.       |

---

# **Summary**

- **Allocation:** Assign Ambulance 1 to Site A (3 trips); Ambulance 2 to Site B (3 trips, last partial); Ambulance 3 to Site C (1 trip).  
- **Expected Lives Saved:** Approximately 5.54 lives under baseline delay and misinformation uncertainties.  
- **Assumptions:** Include stepwise survival, fixed capacity, independent travel delays and misinformation, no bias in patient selection.  
- **Alternate Priors:** Optimistic delays raise expected lives saved (~6.1), increased misinformation reduce it (~4.7).  
- **Real-time Algorithm:** Bayesian updating of beliefs, dynamic scheduling maximizing marginal expected survival gains, integrating drone survey data.  
- **Stress Tests:** Multi-scenario simulation on delays, misinformation, and fairness; report survival and equity metrics.  
- **Ethics & Legal:** Proactively mitigate bias, privacy, and fairness risks with transparency and human oversight.

---

If more detailed numeric optimization or continuous reallocation is feasible in your system, incorporate dynamic re-planning based on real-time data — especially after drone surveys — to further improve lives saved under evolving uncertainties.

---

**This plan balances maximizing expected survival, legal/ethical fairness, and robustness amid significant uncertainties.**

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>